# Proprietà statistiche dei testi generati da LLM lungo uno sweep di temperatura

Analisi di cinque modelli Qwen *base* fatti generare a temperatura crescente
($T = 0.4 \dots 1.5$, passo $0.1$) a partire dallo stesso prompt di 2000 token
tratto da *War and Peace*, con campionamento a sola temperatura
(`top_p = 1`, `top_k = -1`, `repetition_penalty = 1`).

L'obiettivo è misurare **come cambia la struttura statistica del testo fra basse
e alte temperature** e localizzare, per ogni modello, la temperatura alla quale
il testo generato somiglia di più a quello umano.

## Paper di riferimento

| Sigla | Riferimento | Cosa fornisce a questo notebook |
|---|---|---|
| **[Z]** | Mikhaylovskiy, *Zipf's and Heaps' Laws for Tokens and LLM-generated Texts*, Findings of EMNLP 2025 | Istogramma cumulativo di Zipf e relativo MAPE, descrittore $R$, protocollo dello sweep di temperatura, lettura in termini di transizione di fase |
| **[F]** | Mikhaylovskiy, *States of LLM-generated Texts and Phase Transitions between Them*, MathAI 2025 | Le tre fasi (periodica, critica, amorfa), autocorrelazione su embedding, parametro di periodicità $\max\lvert\mathrm{FFT}\rvert$, GAPELMAPER |
| **[L]** | Lippi, Montemurro, Degli Esposti, Cristadoro, *Natural Language Statistical Features of LSTM-generated Texts*, IEEE TNNLS 30(11), 2019 | Fit di Zipf e Heaps, DFA sulle sequenze di caratteri, entropia e divergenza KL simmetrizzata stimate per compressione, LCS |
| **[C]** | Hadad, Loru, Nudo, Di Marco, Cinelli, Quattrociocchi, *The Statistical Signature of LLMs*, 2026 | Batteria di misure basate su compressione: compression ratio, compressione condizionata, curve di prefisso, NCD contro shuffle, entropie normalizzate, distanze di ripetizione |

## Notazione

I quattro paper usano simboli in conflitto fra loro. Qui si adotta una
convenzione unica, indicata di volta in volta insieme a quella originale.

| Grandezza | Simbolo qui | In [Z] | In [L] | In [C] |
|---|---|---|---|---|
| Temperatura di campionamento | $T$ | $t$ | $T$ | — |
| Esponente di Zipf, $f(r) \propto r^{-\alpha}$ | $\alpha$ | $\alpha$ | $\beta$ | — |
| Esponente di Heaps, $w(n) \propto n^{\beta}$ | $\beta$ | $\beta$ | $\nu$ | — |
| Errore del fit power-law sull'istogramma cumulativo | MAPE | MAPE | — | — |
| Tipi nuovi nella seconda metà / nella prima | $R$ | $R$ | — | — |
| Esponente DFA, $F(L) \sim L^{\alpha_{\mathrm{DFA}}}$ | $\alpha_{\mathrm{DFA}}$ | — | $\alpha$ | — |
| Autocorrelazione a distanza $\tau$ | $C(\tau)$ | — | — | — |
| Compression ratio $C(x)/\lvert x\rvert$ | $R_{\mathrm{gzip}}$ | — | — | $R(x)$ |
| Divergenza KL simmetrizzata | $D_s$ | — | $D_s$ | — |

Il simbolo $\alpha$ è riservato a Zipf e $R$ al descrittore di [Z]; l'esponente
DFA e il compression ratio ricevono un pedice per evitare le due collisioni.

## Valori di riferimento sui testi umani

[Z] riporta, sul proprio corpus di sei opere letterarie in cinque lingue
tokenizzate con BPE:

* $\alpha = 1.006 \pm 0.05$
* $\beta = 0.801 \pm 0.02$
* $R = 0.17 \pm 0.05$
* $\mathrm{MAPE} = 0.156 \pm 0.077$

Questi valori sono calcolati su **opere intere**, mentre i testi generati qui
analizzati sono lunghi 24 000 token. Poiché $R$, $\beta$ e MAPE dipendono in
modo marcato dalla lunghezza (§4.6), il confronto principale di questo notebook
usa una **baseline umana misurata sulla stessa lunghezza**; le bande pubblicate
sono riportate nei grafici solo come contesto.

In [ ]:
# =====================================================================
# 0. SETUP
# =====================================================================
import os, re, sys, json, gzip, math, time, hashlib, collections, itertools, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

try:
    from scipy import stats as _sps
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

try:
    import tiktoken
except ImportError:
    raise SystemExit("Manca tiktoken. Installalo con:  pip install tiktoken")

try:
    display
except NameError:
    def display(x): print(x)

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.figsize': (6.6, 4.3),
})
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy={HAS_SCIPY}")

In [ ]:
# =====================================================================
# 1. CONFIGURAZIONE
# =====================================================================
def _trova_radice():
    """Risale dalla posizione del notebook fino alla cartella che contiene i
    dataset .jsonl, così il notebook funziona sia se lanciato dalla sua cartella
    sia dalla radice del progetto."""
    qui = Path.cwd().resolve()
    for cand in [qui, *qui.parents]:
        if list(cand.glob('sweep_qwen*.jsonl')):
            return cand
    raise SystemExit("Non trovo i dataset .jsonl risalendo da " + str(qui))

class CFG:
    RADICE = _trova_radice()                       # 'Dataset totale', SOLA LETTURA
    QUI    = Path.cwd().resolve()                  # 'Analisi Sweep di temperature'
    OUT    = QUI / 'Risultati Sweep'               # tutti gli output vanno qui
    CACHE  = QUI / 'cache'                         # GloVe e testo di riferimento

    # --- registro dei modelli -------------------------------------------------
    # etichetta -> (pattern del file, famiglia, miliardi di parametri, architettura)
    MODELLI = {
        'Qwen2.5-1.5B':    dict(pattern='sweep_qwen2.5-1.5b.jsonl',
                                famiglia='Qwen2.5', params_b=1.54, arch='transformer denso'),
        'Qwen2.5-3B':      dict(pattern='sweep_qwen2.5-3b.jsonl',
                                famiglia='Qwen2.5', params_b=3.09, arch='transformer denso'),
        'Qwen2.5-14B':     dict(pattern='sweep_qwen2.5-14b.jsonl',
                                famiglia='Qwen2.5', params_b=14.7, arch='transformer denso'),
        'Qwen3.5-2B-Base': dict(pattern='sweep_qwen3.5-2b-base.jsonl',
                                famiglia='Qwen3.5', params_b=2.0,  arch='ibrida linear+full'),
        'Qwen3.5-4B-Base': dict(pattern='sweep_qwen3.5-4b-base.jsonl',
                                famiglia='Qwen3.5', params_b=4.0,  arch='ibrida linear+full'),
    }
    ESCLUDI = ('_short', '_failed', '.bak')

    # --- testo umano di riferimento ------------------------------------------
    RIF_NOME  = 'war_and_peace.txt'
    RIF_URL   = 'https://www.gutenberg.org/cache/epub/2600/pg2600.txt'
    RIF_CACHE_ALT = 'corpora_cache/f6c229e9b581.txt'   # copia già presente nel progetto

    # --- unità di analisi ------------------------------------------------------
    # [Z] sec.3 argomenta che a temperatura alta le "parole" non sono più
    # un'unità sensata e che i token sono la scelta naturale. Come in [Z], il
    # tokenizer di analisi è UNICO per tutti i modelli ed ESTERNO alle famiglie
    # che hanno generato i testi (loro usano Mistral tekken su Qwen/Llama/
    # Granite/Mixtral; qui cl100k_base di OpenAI su Qwen2.5/Qwen3.5), così gli
    # esponenti sono confrontabili fra modelli.
    TOKENIZER = 'cl100k_base'
    N_TOK     = 20000       # troncamento comune, in token di analisi
    MIN_TOK   = 2000        # sotto questa soglia il documento viene escluso

    # --- parametri delle analisi ----------------------------------------------
    ZIPF_RANK_WINDOW = (100, 1000)   # [L] sec.V-A: fit fra rango 10^2 e 10^3
    HEAPS_TMIN       = 1000          # [L] sec.V-A: fit nella regione t > 1000
    DFA_LI, DFA_LF, DFA_N = 100, 10000, 12   # [L] sec.V-B: L da 10^2 a 10^4
    KL_CHUNK         = 30000         # caratteri per la stima di entropia e KL
    LCS_CHUNK        = 60000
    LCS_RIF_CHUNK    = 400000
    ACF_SHORT_LAGS   = np.arange(1, 101)     # [F] sec.2.5: lag da 1 a 100
    ACF_LONG_MAX     = 3000                  # [F] sec.2.6
    GAPELMAPER_W     = 600                   # [F] Fig.13: finestra corta
    COPERTURA_MIN    = 0.50   # sotto questa copertura GloVe l'ACF non è valutabile
    SOGLIA_FFT       = 3.0    # sopra: fase periodica
    N_SHUFFLE_DFA    = 2
    SEED             = 20260817

    # --- valori pubblicati in [Z] su testi umani interi ------------------------
    RIF_Z = dict(alpha=(1.006, 0.05), beta=(0.801, 0.02),
                 R=(0.17, 0.05), mape=(0.156, 0.077))

CFG.OUT.mkdir(parents=True, exist_ok=True)
CFG.CACHE.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(CFG.SEED)

print(f"Radice dei dati (sola lettura) : {CFG.RADICE}")
print(f"Cartella del notebook          : {CFG.QUI}")
print(f"Output                         : {CFG.OUT}")

In [ ]:
# =====================================================================
# 1.1 PALETTE
# =====================================================================
# Dodici colori qualitativi ben separati, uno per temperatura, ciascuno con un
# marcatore diverso: i grafici restano leggibili anche stampati in bianco e nero.
COLORI_T = ['#e6194B', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4',
            '#f032e6', '#469990', '#9A6324', '#808000', '#000075', '#808080']
MARKER_T = ['o', 's', '^', 'v', 'D', 'P', 'X', '*', '<', '>', 'h', 'p']

COLORI_MODELLO = {'Qwen2.5-1.5B': '#1b9e77', 'Qwen2.5-3B': '#d95f02',
                  'Qwen2.5-14B': '#7570b3', 'Qwen3.5-2B-Base': '#e7298a',
                  'Qwen3.5-4B-Base': '#66a61e'}
MARKER_MODELLO = {'Qwen2.5-1.5B': 'o', 'Qwen2.5-3B': 's', 'Qwen2.5-14B': '^',
                  'Qwen3.5-2B-Base': 'D', 'Qwen3.5-4B-Base': 'v'}
COLORE_UMANO = '#000000'

def stile_T(T, tutte):
    i = list(np.round(tutte, 3)).index(round(float(T), 3))
    return dict(color=COLORI_T[i % len(COLORI_T)], marker=MARKER_T[i % len(MARKER_T)])

def stile_modello(m):
    return dict(color=COLORI_MODELLO.get(m, '#333333'),
                marker=MARKER_MODELLO.get(m, 'o'))

def salva(nome):
    """Tutte le figure finiscono nell'unica cartella 'Risultati Sweep'."""
    for ext in ('png', 'pdf'):
        plt.savefig(CFG.OUT / f'{nome}.{ext}')
    plt.show()

def banda_riferimento(ax, chiave, etichetta):
    """Banda orizzontale con il valore pubblicato in [Z] su opere intere."""
    if chiave not in CFG.RIF_Z:
        return
    mu, sd = CFG.RIF_Z[chiave]
    ax.axhspan(mu - sd, mu + sd, color='#999999', alpha=.16, zorder=0)
    ax.axhline(mu, color='#555555', ls='-.', lw=1.1, zorder=1, label=etichetta)

fig, axes = plt.subplots(1, 2, figsize=(13, 1.8))
demo = [round(0.4 + i * 0.1, 1) for i in range(12)]
for i, T in enumerate(demo):
    axes[0].scatter([i], [0], s=170, **stile_T(T, demo))
    axes[0].text(i, .38, f'{T:g}', ha='center', fontsize=8)
axes[0].set_ylim(-.6, .85); axes[0].axis('off')
axes[0].set_title('Temperature', fontsize=10)
for i, (m, c) in enumerate(COLORI_MODELLO.items()):
    axes[1].scatter([i], [0], s=200, color=c, marker=MARKER_MODELLO[m])
    axes[1].text(i, .38, m.replace('Qwen', ''), ha='center', fontsize=7.5)
axes[1].set_ylim(-.6, .85); axes[1].axis('off')
axes[1].set_title('Modelli', fontsize=10)
plt.tight_layout(); salva('00_palette')

## 2. Caricamento dei dataset

I file vengono cercati per pattern nella cartella radice del progetto, che viene
trattata come **sola lettura**: questo notebook non scrive nulla al di fuori
della propria cartella.

In [ ]:
def trova_dataset():
    trovati = {}
    for etichetta, info in CFG.MODELLI.items():
        cand = [p for p in sorted(CFG.RADICE.glob(info['pattern']))
                if not any(x in p.name for x in CFG.ESCLUDI)]
        if not cand:
            print(f"  [ ] {etichetta:16s} nessun file per '{info['pattern']}'")
            continue
        f = max(cand, key=lambda p: p.stat().st_mtime)
        trovati[etichetta] = f
        print(f"  [x] {etichetta:16s} {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
        for altro in cand:
            if altro != f:
                print(f"      [!] ignorato (più vecchio): {altro.name}")
    return trovati

print("Dataset trovati:\n")
FILE_MODELLI = trova_dataset()
MODELLI = [m for m in CFG.MODELLI if m in FILE_MODELLI]
if not MODELLI:
    raise SystemExit("Nessun dataset trovato.")

DOCS = []
for etichetta, path in FILE_MODELLI.items():
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            r['_modello'] = etichetta
            DOCS.append(r)

META = pd.DataFrame([{k: v for k, v in r.items()
                      if k not in ('generated_text', 'eos_positions',
                                   'logit_bias_applied', 'stop_token_ids')}
                     for r in DOCS])
META['n_char'] = [len(r['generated_text']) for r in DOCS]

print(f"\nDocumenti totali: {len(DOCS)}\n")
display(META.groupby('_modello').agg(
    n=('temperature', 'count'),
    T_min=('temperature', 'min'), T_max=('temperature', 'max'),
    n_T=('temperature', 'nunique'),
    per_T_min=('temperature', lambda s: s.value_counts().min()),
    per_T_max=('temperature', lambda s: s.value_counts().max()),
    char_mediani=('n_char', 'median')))

print("\nCampioni per (modello, temperatura) — le celle con 1 solo campione")
print("non avranno barra d'errore:")
display(META.pivot_table(index='temperature', columns='_modello',
                         values='sample_id', aggfunc='count').fillna(0).astype(int))

### 2.1 Verifica del protocollo di generazione

Prima di confrontare i modelli occorre accertarsi che i dataset siano stati
prodotti con lo stesso protocollo. Contano soprattutto: il prompt di partenza,
`max_tokens`, la lunghezza effettiva della generazione e i parametri di
campionamento — [Z] §4.1 richiede esplicitamente **nessun top-$k$, nessun
top-$p$, nessuna penalità di ripetizione**, perché l'unica variabile in gioco
deve essere la temperatura.

In [ ]:
def verifica_protocollo():
    campi = ['prompt_sha256_16', 'prompt_length_chars', 'prompt_length_tokens',
             'max_tokens', 'top_p', 'top_k', 'repetition_penalty',
             'generation_mode', 'completion_tokens', 'finish_reason', 'tokenizer_id']
    righe = []
    for m in MODELLI:
        recs = [r for r in DOCS if r['_modello'] == m]
        riga = {'modello': m, 'n': len(recs)}
        for c in campi:
            vals = {r[c] for r in recs if c in r and r[c] is not None}
            riga[c] = ('—' if not vals else
                       (list(vals)[0] if len(vals) == 1 else
                        f'{min(vals)}–{max(vals)}' if all(isinstance(v, (int, float)) for v in vals)
                        else f'{len(vals)} valori'))
        righe.append(riga)
    display(pd.DataFrame(righe).set_index('modello').T)

    print("Esito dei controlli:")
    # 1) campionamento a sola temperatura
    tp = {r.get('top_p') for r in DOCS}; tk = {r.get('top_k') for r in DOCS}
    rp = {r.get('repetition_penalty') for r in DOCS}
    if tp == {1.0} and tk == {-1} and rp == {1.0}:
        print("  [OK] campionamento a sola temperatura (top_p=1, top_k=-1, "
              "repetition_penalty=1), come richiesto da [Z] sec.4.1")
    else:
        print(f"  [!!] parametri di decoding non neutri: top_p={tp} top_k={tk} "
              f"rep_pen={rp} — il confronto con [Z] non è diretto")
    # 2) lunghezza
    ct = {r.get('completion_tokens') for r in DOCS if r.get('completion_tokens')}
    if not ct:
        print("  [i] campo completion_tokens assente: la lunghezza viene "
              "controllata direttamente sui token di analisi nel §3")
    elif len(ct) == 1:
        print(f"  [OK] tutti i campioni hanno {list(ct)[0]} token di completion: "
              "nessun confounder di lunghezza")
    else:
        print(f"  [!] lunghezza non uniforme ({min(ct)}–{max(ct)} token): il "
              f"troncamento comune a CFG.N_TOK={CFG.N_TOK} è indispensabile")
    # 3) prompt
    off = {(r.get('prompt_sha256_16'), r.get('prompt_length_chars')) for r in DOCS}
    lung = {l for _, l in off}
    if len(off) == 1:
        print(f"  [OK] prompt identico in tutti i dataset ({list(off)[0][0]})")
    else:
        print(f"  [i] {len(off)} varianti del prompt registrate, lunghezze {sorted(lung)} "
              f"caratteri: differiscono solo per dove cade il confine dei 2000 token")
        print("      con i diversi tokenizer Qwen2.5/Qwen3.5. Il §2.2 verifica che")
        print("      tutte partano dallo stesso punto del romanzo.")
    # 4) esito della generazione
    fr = collections.Counter(r.get('finish_reason') for r in DOCS)
    sc = collections.Counter(r.get('success') for r in DOCS)
    print(f"  [i] finish_reason: {dict(fr)} | success: {dict(sc)}")
    nc = [r.get('n_continuations') for r in DOCS if r.get('n_continuations') is not None]
    if nc:
        print(f"  [i] modalità 'continuation': i testi sono ricuciti dopo ogni EOS, "
              f"da 0 a {max(nc)} riprese (mediana {int(np.median(nc))}). "
              "L'effetto è controllato nel §5.2.")

verifica_protocollo()

### 2.2 Testo umano di riferimento

Il prompt di generazione è un blocco di 2000 token estratto da *War and Peace*.
La baseline umana corretta non è l'inizio del romanzo, ma **la continuazione
dello stesso passaggio**: così il confronto è continuazione umana contro
continuazione generata, a parità di contesto.

Il punto di aggancio non viene cercato a mano: si prende l'hash SHA-256
registrato nei dataset e si individua nel romanzo la finestra che lo riproduce.
Se gli hash dei vari dataset puntano tutti allo stesso offset, i modelli sono
partiti dallo stesso punto anche quando i conteggi di caratteri differiscono.

In [ ]:
def carica_riferimento():
    p = CFG.CACHE / CFG.RIF_NOME
    if not p.exists():
        alt = CFG.RADICE / CFG.RIF_CACHE_ALT
        if alt.exists():
            print(f"Uso la copia già presente nel progetto: {alt.name}")
            p.write_bytes(alt.read_bytes())
        else:
            print("Scarico War and Peace da Project Gutenberg...")
            import urllib.request
            urllib.request.urlretrieve(CFG.RIF_URL, p)
    return p.read_bytes().decode('utf-8', errors='replace')

ROMANZO = carica_riferimento()
print(f"Romanzo completo: {len(ROMANZO):,} caratteri\n")

def localizza_prompt(romanzo, max_offset=400000):
    """Trova, per ogni (hash, lunghezza) registrato nei dataset, l'offset in
    caratteri nel romanzo che lo riproduce. Restituisce {(hash, len): offset}.

    Per non ricodificare in UTF-8 una finestra da 8 KB a ogni tentativo, il
    romanzo viene codificato una volta sola e si costruisce la mappa
    carattere -> offset in byte; il confronto avviene poi su fette di byte."""
    voluti = {(r.get('prompt_sha256_16'), r.get('prompt_length_chars'))
              for r in DOCS if r.get('prompt_sha256_16') and r.get('prompt_length_chars')}
    testa = romanzo[:max_offset + max(L for _, L in voluti) + 10]
    byte_di = np.zeros(len(testa) + 1, dtype=np.int64)
    byte_di[1:] = np.cumsum([len(ch.encode('utf-8')) for ch in testa])
    grezzo = testa.encode('utf-8')
    trovati = {}
    for h, L in voluti:
        for off in range(min(max_offset, len(testa) - L)):
            if hashlib.sha256(grezzo[byte_di[off]:byte_di[off + L]]).hexdigest()[:16] == h:
                trovati[(h, L)] = off
                break
    return trovati

OFFSET = localizza_prompt(ROMANZO)
for (h, L), off in sorted(OFFSET.items(), key=lambda kv: kv[1]):
    mods = sorted({r['_modello'] for r in DOCS
                   if r.get('prompt_sha256_16') == h and r.get('prompt_length_chars') == L})
    print(f"  {h}  {L:>5} char  ->  offset {off:>6}   {', '.join(m.replace('Qwen','') for m in mods)}")

if not OFFSET:
    raise SystemExit("Nessun prompt localizzato nel romanzo: controlla il file di riferimento.")
if len({off for off in OFFSET.values()}) == 1:
    print("\n  [OK] tutti i prompt partono dallo stesso punto del romanzo.")
else:
    print("\n  [!!] i prompt partono da punti DIVERSI: il confronto fra modelli va rivisto.")

# La continuazione umana comincia dove finisce il prompt più lungo.
FINE_PROMPT = max(off + L for (h, L), off in OFFSET.items())
UMANO_TESTO = ROMANZO[FINE_PROMPT:]
for marker in ('*** END OF', '***END OF', 'End of the Project Gutenberg'):
    j = UMANO_TESTO.find(marker)
    if j != -1:
        UMANO_TESTO = UMANO_TESTO[:j]
        break
print(f"\nPrompt: caratteri {min(OFFSET.values()):,}–{FINE_PROMPT:,} del romanzo")
print(f"Continuazione umana disponibile: {len(UMANO_TESTO):,} caratteri")
print(f"Inizia con: {UMANO_TESTO[:80]!r}")

## 3. Tokenizzazione

Due livelli, con ruoli diversi:

* **token BPE** — livello **primario**. [Z] §3 lo motiva così: alle temperature
  alte i modelli producono sequenze che non sono interpretabili come parole, e
  un'analisi lessicale misurerebbe artefatti di segmentazione invece che
  proprietà del testo. Come in [Z], il tokenizer è **unico per tutti i modelli**
  ed **esterno** alle famiglie che hanno generato i testi, in modo che gli
  esponenti siano confrontabili fra modelli;
* **parole** — livello secondario (`\w+` su testo minuscolo, cifre rimosse),
  usato solo per le misure che [L] e [C] definiscono sulle parole: TTR, entropia
  a livello di parola, distanze di ripetizione, $n$-grammi ripetuti, e per la
  mappatura su GloVe del §11.

La DFA fa eccezione: [L] §III-B la definisce a livello di **carattere**,
punteggiatura e maiuscole incluse, e qui si segue quella definizione.

In [ ]:
ENC = tiktoken.get_encoding(CFG.TOKENIZER)
print(f"Tokenizer di analisi: {CFG.TOKENIZER}  (vocabolario {ENC.n_vocab:,})")

def tokenizza(testo, n=None):
    """Token BPE troncati a n, e il testo corrispondente esatto."""
    ids = ENC.encode(testo, disallowed_special=())
    if n:
        ids = ids[:n]
    return [str(i) for i in ids], ENC.decode(ids)

def parole(testo):
    return re.findall(r'\w+', re.sub(r'\d', ' ', testo.lower()))

# --- lunghezza disponibile prima di fissare il troncamento -------------------
n_disp = [len(ENC.encode(r['generated_text'], disallowed_special=())) for r in DOCS]
print(f"Token di analisi per documento: min {min(n_disp):,}  "
      f"mediana {int(np.median(n_disp)):,}  max {max(n_disp):,}")
if min(n_disp) < CFG.N_TOK:
    print(f"  [!] {sum(1 for x in n_disp if x < CFG.N_TOK)} documenti sotto "
          f"CFG.N_TOK={CFG.N_TOK:,}: verranno usati per intero e segnalati.")
else:
    print(f"  [OK] tutti i documenti superano CFG.N_TOK={CFG.N_TOK:,}.")

t0 = time.time()
for i, r in enumerate(DOCS):
    tok, txt = tokenizza(r['generated_text'], CFG.N_TOK)
    r['_tok'], r['_testo'] = tok, txt
    r['_parole'] = parole(txt)
    r['_valido'] = len(tok) >= CFG.MIN_TOK
print(f"Tokenizzati {len(DOCS)} documenti in {time.time()-t0:.0f}s")

# --- baseline umana, stesso trattamento --------------------------------------
UMANO_TOK, UMANO_TXT = tokenizza(UMANO_TESTO, CFG.N_TOK)
UMANO_PAROLE = parole(UMANO_TXT)
print(f"\nBaseline umana: {len(UMANO_TOK):,} token, {len(UMANO_TXT):,} caratteri, "
      f"{len(UMANO_PAROLE):,} parole")

TEMPERATURE = sorted({r['temperature'] for r in DOCS})
print(f"Temperature: {TEMPERATURE}")
VALIDI = [r for r in DOCS if r['_valido']]
print(f"Documenti validi: {len(VALIDI)}/{len(DOCS)}")

## 4. Funzioni di analisi

Ogni blocco indica il paper e la sezione da cui la definizione proviene.

In [ ]:
# =====================================================================
# 4.1 ZIPF   [Z] sec.3-4.3, [L] sec.III-A
# =====================================================================
def zipf_rango_frequenza(tokens):
    c = collections.Counter(tokens)
    f = np.array([n for _, n in c.most_common()], dtype=float)
    return np.arange(1, len(f) + 1, dtype=float), f

def fit_powerlaw(x, y):
    """Fit ai minimi quadrati di y ~ x^(-a) in coordinate log-log.
    Restituisce (a, intercetta, MAPE). [Z] usa esattamente questo stimatore,
    motivando in sec.3 la scelta dei minimi quadrati rispetto alla massima
    verosimiglianza (instabile rispetto al valore iniziale)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = (x > 0) & (y > 0)
    if m.sum() < 3:
        return np.nan, np.nan, np.nan
    lx, ly = np.log10(x[m]), np.log10(y[m])
    slope, inter = np.polyfit(lx, ly, 1)
    yhat = 10.0 ** (inter + slope * lx)
    return -slope, inter, float(np.mean(np.abs((y[m] - yhat) / y[m])))

def zipf_alpha_rango(tokens, finestra=None):
    """[L] sec.V-A: esponente misurato fra rango 10^2 e 10^3."""
    lo, hi = finestra or CFG.ZIPF_RANK_WINDOW
    r, f = zipf_rango_frequenza(tokens)
    m = (r >= lo) & (r <= min(hi, r.max()))
    if m.sum() < 5:
        return np.nan
    a, _, _ = fit_powerlaw(r[m], f[m])
    return a

def istogramma_cumulativo(tokens):
    """[Z] sec.3: istogramma cumulativo, P(occorrenze > x) in funzione di x.
    È la rappresentazione che [Z] preferisce al classico rango-frequenza."""
    c = collections.Counter(tokens)
    conteggi = np.array(sorted(c.values()), dtype=float)
    x = np.unique(conteggi)
    surv = np.array([(conteggi > xi).sum() for xi in x], dtype=float) / len(conteggi)
    m = surv > 0
    return x[m], surv[m]

def zipf_cumulativo_fit(tokens):
    """[Z]: esponente alpha e MAPE del fit power-law sull'istogramma cumulativo.
    Il MAPE è il descrittore centrale di [Z] sec.4.3: il suo minimo in funzione
    di T individua lo stato critico."""
    x, y = istogramma_cumulativo(tokens)
    if len(x) < 5:
        return np.nan, np.nan
    a, _, mape = fit_powerlaw(x, y)
    return a, mape

# =====================================================================
# 4.2 HEAPS E DESCRITTORE R   [Z] sec.4.2, [L] sec.III-A
# =====================================================================
def heaps_curva(tokens):
    seen, out, n = set(), np.empty(len(tokens), dtype=np.int64), 0
    for i, w in enumerate(tokens):
        if w not in seen:
            seen.add(w); n += 1
        out[i] = n
    return out

def heaps_beta(tokens, tmin=None):
    """[L] sec.V-A: fit della curva n(t) nella regione t > 1000."""
    tmin = tmin or CFG.HEAPS_TMIN
    n = heaps_curva(tokens)
    if len(n) < tmin + 100:
        tmin = max(10, len(n) // 10)
    idx = np.unique(np.logspace(np.log10(tmin), np.log10(len(n)), 60).astype(int))
    idx = idx[(idx >= tmin) & (idx <= len(n))]
    if len(idx) < 5:
        return np.nan
    a, _, _ = fit_powerlaw(idx, n[idx - 1])
    return -a

def descrittore_R(tokens):
    """[Z] sec.4.2: rapporto fra il numero di token che compaiono per la prima
    volta nella SECONDA metà del testo e quelli che compaiono per la prima volta
    nella PRIMA metà, cioè |V2 \\ V1| / |V1|.

    [Z] propone R perché la curva di Heaps dei testi generati è tipicamente così
    lontana da una legge di potenza che fittarne l'esponente non ha senso; R è
    invece sempre definito. Valore medio sui testi naturali interi: 0.17 +- 0.05."""
    if len(tokens) < 20:
        return np.nan
    h = len(tokens) // 2
    V1, V2 = set(tokens[:h]), set(tokens[h:])
    return len(V2 - V1) / len(V1) if V1 else np.nan

def ttr(tokens):
    return len(set(tokens)) / len(tokens) if len(tokens) else np.nan

def ngram_ripetuti(tokens, n=8):
    """Quota di n-grammi non unici: indicatore diretto di degenerazione."""
    if len(tokens) < 3 * n:
        return np.nan
    g = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    return 1.0 - len(set(g)) / len(g)

def distanze_ripetizione(tokens):
    """[C]: media e deviazione standard della distanza fra occorrenze
    consecutive dello stesso tipo."""
    pos = collections.defaultdict(list)
    for i, w in enumerate(tokens):
        pos[w].append(i)
    d = [np.diff(v) for v in pos.values() if len(v) > 1]
    if not d:
        return np.nan, np.nan
    a = np.concatenate(d)
    return float(a.mean()), float(a.std())

print("4.1-4.2 Zipf, Heaps, R: caricate.")

In [ ]:
# =====================================================================
# 4.3 DFA   [L] sec.III-B
# =====================================================================
def serie_rango_caratteri(testo):
    """[L] sec.III-B: la mappatura è a livello di CARATTERE — punteggiatura,
    cifre, maiuscole e lettere accentate incluse — e ogni carattere viene
    sostituito dal proprio rango di frequenza (il più frequente riceve 1)."""
    c = collections.Counter(testo)
    rank = {ch: i + 1 for i, (ch, _) in enumerate(c.most_common())}
    return np.fromiter((rank[ch] for ch in testo), dtype=np.float64, count=len(testo))

def _dfa_F(serie, L, ordine=1):
    """[L] eq.(9)-(10): random walk cumulativo, detrending lineare per finestre
    di lunghezza L, fluttuazione quadratica media."""
    x = np.cumsum(serie - serie.mean())
    nseg = len(x) // L
    if nseg < 1:
        return np.nan
    seg = x[:nseg * L].reshape(nseg, L)
    t = np.linspace(-1, 1, L)
    A = np.vander(t, N=ordine + 1, increasing=True)
    coef, *_ = np.linalg.lstsq(A, seg.T, rcond=None)
    resid = seg.T - A @ coef
    return float(np.sqrt(np.mean(resid ** 2)))

def dfa(serie, li=None, lf=None, n=None, ordine=1):
    """Scale spaziate logaritmicamente da li a lf. Le scale oltre N/4 vengono
    scartate: con meno di quattro segmenti la fluttuazione media non è stimabile
    in modo affidabile."""
    li = li or CFG.DFA_LI; lf = lf or CFG.DFA_LF; n = n or CFG.DFA_N
    dl = (np.log2(lf) - np.log2(li)) / (n - 1)
    Ls = np.unique(np.round(li * 2 ** (dl * np.arange(n)))).astype(int)
    Ls = Ls[Ls <= max(len(serie) // 4, li)]
    F = np.array([_dfa_F(serie, int(L), ordine) for L in Ls])
    m = np.isfinite(F) & (F > 0)
    return Ls[m], F[m]

def dfa_alpha(serie, **kw):
    """[L] sec.III-B: F(L) ~ L^alpha. alpha = 0.5 per sequenze non correlate,
    alpha > 0.5 per correlazioni persistenti a lungo raggio."""
    L, F = dfa(serie, **kw)
    if len(L) < 3:
        return np.nan
    return float(np.polyfit(np.log(L), np.log(F), 1)[0])

def dfa_alpha_mescolato(serie, n=None, rng=None):
    """Null model di [L] Fig.4: la stessa serie mescolata deve dare alpha ~ 0.5."""
    n = n or CFG.N_SHUFFLE_DFA
    rng = rng or RNG
    out = []
    for _ in range(n):
        s = np.array(serie, copy=True); rng.shuffle(s)
        out.append(dfa_alpha(s))
    return float(np.nanmean(out))

# --- validazione dell'implementazione su segnali con esponente noto ----------
_rng = np.random.default_rng(0)
_bianco = _rng.normal(size=200000)
_walk = np.cumsum(_rng.normal(size=200000))
_ver = pd.DataFrame([
    dict(segnale='rumore bianco', atteso=0.5, ottenuto=round(dfa_alpha(_bianco), 3)),
    dict(segnale='random walk',   atteso=1.5, ottenuto=round(dfa_alpha(_walk), 3)),
])
_ver['scarto'] = (_ver.ottenuto - _ver.atteso).abs().round(3)
_ver['esito'] = np.where(_ver.scarto < 0.05, 'PASS', 'FAIL')
display(_ver)
print("4.3 DFA: caricata e validata." if (_ver.esito == 'PASS').all()
      else "[!] 4.3 DFA: validazione FALLITA.")

In [ ]:
# =====================================================================
# 4.4 COMPRESSIONE   [C] sec.3.2
# =====================================================================
def gzip_ratio(testo):
    """[C] eq.(4): R(x) = C(x)/|x|, con gzip su byte UTF-8 grezzi.
    Valori bassi = testo molto regolare e quindi molto comprimibile."""
    b = testo.encode('utf-8')
    return len(gzip.compress(b, 9)) / len(b) if b else np.nan

def compressione_condizionata(testo):
    """[C] sec.4.1.1: costo incrementale di comprimere la seconda metà data la
    prima, (C(x+y) - C(x)) / |y|. Proxy della prevedibilità dal contesto."""
    h = len(testo) // 2
    x, y = testo[:h], testo[h:]
    if not y:
        return np.nan
    cx = len(gzip.compress(x.encode('utf-8'), 9))
    cxy = len(gzip.compress((x + y).encode('utf-8'), 9))
    return (cxy - cx) / len(y.encode('utf-8'))

def ncd(a, b):
    """Normalized Compression Distance (Li et al. 2004), usata in [C]."""
    ca = len(gzip.compress(a.encode('utf-8'), 9))
    cb = len(gzip.compress(b.encode('utf-8'), 9))
    cab = len(gzip.compress((a + b).encode('utf-8'), 9))
    return (cab - min(ca, cb)) / max(ca, cb)

def dividi_frasi(testo):
    return [u for u in re.split(r'(?<=[.!?])\s+', testo) if u.strip()]

def curva_prefissi(testo, n_punti=25):
    """[C] sec.3.2: il compression ratio ricalcolato su prefissi via via più
    lunghi, misurati in numero di frasi. È la misura della Fig.1C di [C]:
    nei testi LLM la regolarità si accumula con la lunghezza, in quelli umani no."""
    unita = dividi_frasi(testo)
    if len(unita) < 5:
        return np.array([]), np.array([])
    ks = np.unique(np.linspace(1, len(unita), n_punti).astype(int))
    xs, ys = [], []
    for k in ks:
        p = ' '.join(unita[:k])
        if len(p) >= 50:
            xs.append(k); ys.append(gzip_ratio(p))
    return np.array(xs, float), np.array(ys, float)

def statistiche_prefissi(testo):
    """[C] sec.4.1.1: media, pendenza (prefix ratio trend) e variabilità
    (prefix ratio variability) della curva di prefisso."""
    x, y = curva_prefissi(testo)
    if len(x) < 5:
        return np.nan, np.nan, np.nan
    return float(y.mean()), float(np.polyfit(x, y, 1)[0]), float(y.std())

def mescola_dentro_frasi(testo, rng):
    out = []
    for s in dividi_frasi(testo):
        w = s.split()
        rng.shuffle(w)
        out.append(' '.join(w))
    return ' '.join(out)

def contributo_ordine(testo, rng):
    """[C] sec.4.1.1 'word-order contribution metrics': si permutano le parole
    dentro i confini di frase e si confrontano originale e permutato con
    (i) il gap del compression ratio e (ii) la NCD."""
    sh = mescola_dentro_frasi(testo, rng)
    if not sh:
        return np.nan, np.nan
    return gzip_ratio(sh) - gzip_ratio(testo), ncd(testo, sh)

def entropia_normalizzata(seq):
    """[C] sec.4.1.1: entropia di Shannon normalizzata su log(numero di tipi).
    Misura quanto uniformemente i simboli sono distribuiti, non la struttura."""
    if not len(seq):
        return np.nan
    c = collections.Counter(seq)
    p = np.array(list(c.values()), dtype=float) / len(seq)
    if len(c) < 2:
        return 0.0
    return float(-(p * np.log(p)).sum() / np.log(len(c)))

print("4.4 Compressione: caricata.")

In [ ]:
# =====================================================================
# 4.5 ENTROPIA, DIVERGENZA KL, LCS   [L] sec.III-C, III-D
# =====================================================================
# [L] stima entropia e KL con algoritmi di compressione della famiglia
# Lempel-Ziv. Qui si usa il cross-parsing di Ziv-Merhav (1993), che è il metodo
# citato in [L] come rif.[38] per la cross-entropia. L'implementazione si appoggia
# a un automa dei suffissi, che riconosce tutte le sottostringhe del riferimento
# in tempo lineare.
class AutomaSuffissi:
    __slots__ = ('next', 'link', 'length', 'last', 'size')

    def __init__(self, s=None):
        self.next = [{}]; self.link = [-1]; self.length = [0]
        self.last = 0; self.size = 1
        if s:
            for ch in s:
                self.extend(ch)

    def extend(self, ch):
        cur = self.size
        self.next.append({}); self.link.append(-1)
        self.length.append(self.length[self.last] + 1); self.size += 1
        p = self.last
        while p != -1 and ch not in self.next[p]:
            self.next[p][ch] = cur; p = self.link[p]
        if p == -1:
            self.link[cur] = 0
        else:
            q = self.next[p][ch]
            if self.length[p] + 1 == self.length[q]:
                self.link[cur] = q
            else:
                clone = self.size
                self.next.append(dict(self.next[q])); self.link.append(self.link[q])
                self.length.append(self.length[p] + 1); self.size += 1
                while p != -1 and self.next[p].get(ch) == q:
                    self.next[p][ch] = clone; p = self.link[p]
                self.link[q] = clone; self.link[cur] = clone
        self.last = cur

    def match_piu_lungo(self, s, start):
        v, l, i, n = 0, 0, start, len(s)
        while i < n:
            nx = self.next[v].get(s[i])
            if nx is None:
                break
            v = nx; l += 1; i += 1
        return l

def _conta_frasi(x, automa):
    n, i, c = len(x), 0, 0
    while i < n:
        i += automa.match_piu_lungo(x, i) + 1
        c += 1
    return c

def entropia_incrociata(x, y):
    """h(x|y) in bit per simbolo: numero di frasi in cui x si decompone rispetto
    al dizionario delle sottostringhe di y, per log2|y|, diviso |x|."""
    if not x or not y:
        return np.nan
    return _conta_frasi(x, AutomaSuffissi(y)) * math.log2(len(y)) / len(x)

def kl(x, y, split=0.5):
    """D(x||y) = h(x2|y1) - h(x2|x1), con x diviso in x1|x2 e y1 preso lungo
    quanto x1. L'appaiamento delle lunghezze dei riferimenti è ciò che rende la
    stima non distorta: con y = x il risultato è esattamente 0."""
    if not x or not y:
        return np.nan
    k = int(len(x) * split)
    x1, x2 = x[:k], x[k:]
    if not x1 or not x2:
        return np.nan
    y1 = y[:len(x1)]
    if len(y1) < len(x1) * 0.5:
        return np.nan
    return entropia_incrociata(x2, y1) - entropia_incrociata(x2, x1)

def kl_simmetrica(x, y, split=0.5):
    """[L] sec.III-C: D_s(X,Y) = (D(X|Y) + D(Y|X)) / 2."""
    d1, d2 = kl(x, y, split), kl(y, x, split)
    if not (np.isfinite(d1) and np.isfinite(d2)):
        return np.nan
    return (d1 + d2) / 2

def tasso_entropia(x, split=0.5):
    """[L] sec.III-C: entropia in bit per carattere, stimata per compressione."""
    if len(x) < 200:
        return np.nan
    k = int(len(x) * split)
    return entropia_incrociata(x[k:], x[:k])

def lcs(a, b, cap=None):
    """[L] sec.III-D: lunghezza della più lunga sottostringa comune, misura di
    plagio dal corpus di riferimento. O(|a| + |b|) via automa dei suffissi."""
    cap = cap or CFG.LCS_RIF_CHUNK
    a, b = a[:cap], b[:cap]
    if not a or not b:
        return 0
    sa = AutomaSuffissi(a)
    v = l = best = 0
    for ch in b:
        while v and ch not in sa.next[v]:
            v = sa.link[v]; l = sa.length[v]
        if ch in sa.next[v]:
            v = sa.next[v][ch]; l += 1
        else:
            v = l = 0
        if l > best:
            best = l
    return best

# --- autotest dello stimatore ------------------------------------------------
def autotest_kl(N=20000):
    AL = 'abcdefghijklmnopqrstuvwxyz '
    def catena(ordine, seed, conc=3.0):
        r = np.random.default_rng(seed)
        ctxs = [''.join(c) for c in itertools.product(AL, repeat=ordine)]
        P = r.dirichlet(np.full(len(AL), 1.0 / conc), size=len(ctxs))
        return {c: P[i] for i, c in enumerate(ctxs)}
    def campiona(ch, n, ordine, seed):
        r = np.random.default_rng(seed)
        out = list(r.choice(list(AL), size=ordine))
        for _ in range(n):
            out.append(r.choice(list(AL), p=ch[''.join(out[-ordine:])]))
        return ''.join(out)
    A, B = catena(2, 1), catena(2, 2)
    a1, a2, b1 = campiona(A, N, 2, 10), campiona(A, N, 2, 11), campiona(B, N, 2, 12)
    u = ''.join(np.random.default_rng(3).choice(list(AL), size=N))
    deg = 'k' * N
    d_id, d_ug = kl(a1, a1), kl(a1, a2)
    d_div, d_un, d_deg = kl(a1, b1), kl(a1, u), kl(a1, deg)
    prove = [("D(x||x) esattamente 0",        d_id,  abs(d_id) < 1e-12),
             ("stessa sorgente, D ~ 0",       d_ug,  abs(d_ug) < 0.15),
             ("sorgente diversa > stessa",    d_div, d_div > d_ug + 0.2),
             ("rumore uniforme > stessa",     d_un,  d_un > d_ug + 0.2),
             ("riferimento degenere enorme",  d_deg, d_deg > 5 * max(d_div, 0.1))]
    righe, ok_tot = [], True
    for nome, val, ok in prove:
        ok = bool(np.isfinite(val) and ok); ok_tot &= ok
        righe.append(dict(test=nome, valore=round(float(val), 5),
                          esito='PASS' if ok else 'FAIL'))
    display(pd.DataFrame(righe))
    return ok_tot

print("4.5 Entropia, KL, LCS: caricate. Autotest dello stimatore:")
KL_OK = autotest_kl()
print("Stimatore KL validato." if KL_OK else "[!] Autotest FALLITO.")

In [ ]:
# =====================================================================
# 4.6 FASI DEL TESTO   [F] sec.1.3, 2.5, 2.6
# =====================================================================
def traiettoria(parole_doc, emb, dim):
    """[F] sec.2.3: ogni parola diventa il suo vettore GloVe; il sistema viene
    centrato sottraendo la media sull'intero testo. Alle parole senza vettore
    corrispondente [F] assegna il vettore nullo DOPO la centratura, e assume che
    tutte le loro correlazioni valgano zero. Le posizioni restano al loro posto,
    così la struttura dei lag è preservata."""
    M = np.zeros((len(parole_doc), dim), dtype=np.float32)
    noti = np.zeros(len(parole_doc), dtype=bool)
    for i, w in enumerate(parole_doc):
        v = emb.get(w)
        if v is not None:
            M[i] = v; noti[i] = True
    if noti.any():
        M[noti] -= M[noti].mean(0)
        M[~noti] = 0.0
    return M, noti

def acf_coseno(M, lags):
    """[F] eq.(6): C(tau) = media della similarità coseno fra i vettori a
    distanza tau. La media è su tutte le N-tau coppie; quelle che coinvolgono un
    vettore nullo contribuiscono zero, come prescritto in [F] sec.2.3."""
    norme = np.linalg.norm(M, axis=1)
    out = np.full(len(lags), np.nan)
    N = len(M)
    for j, tau in enumerate(lags):
        tau = int(tau)
        if tau < 1 or tau >= N:
            continue
        den = norme[:-tau] * norme[tau:]
        num = np.einsum('ij,ij->i', M[:-tau], M[tau:])
        with np.errstate(divide='ignore', invalid='ignore'):
            c = np.where(den > 0, num / np.where(den > 0, den, 1.0), 0.0)
        out[j] = float(c.mean())
    return out

def parametro_periodicita(acf):
    """[F] sec.2.5: massimo modulo della trasformata di Fourier discreta
    dell'ACF normalizzata, escluso il primo coefficiente. È il parametro
    d'ordine della fase periodica: [F] Fig.7-8 lo vede crollare a T ~ 0.8."""
    a = np.nan_to_num(np.asarray(acf, float) - np.nanmean(acf))
    if len(a) < 4:
        return np.nan
    F = np.abs(np.fft.rfft(a))
    return float(F[1:].max()) if len(F) > 1 else np.nan

def gapelmaper(lags, acf):
    """[F] sec.2.6: GAPELMAPER = MAPE(fit power-law) / MAPE(fit esponenziale)
    sull'ACF. Sotto 1 il decadimento è a legge di potenza (struttura gerarchica,
    stato critico); sopra 1 è esponenziale (fase amorfa)."""
    lags, acf = np.asarray(lags, float), np.asarray(acf, float)
    m = np.isfinite(acf) & (acf > 0) & (lags > 0)
    if m.sum() < 6:
        return np.nan
    x, y = lags[m], acf[m]
    yp = np.exp(np.polyval(np.polyfit(np.log(x), np.log(y), 1), np.log(x)))
    ye = np.exp(np.polyval(np.polyfit(x, np.log(y), 1), x))
    mp = np.mean(np.abs((y - yp) / y))
    me = np.mean(np.abs((y - ye) / y))
    return float(mp / me) if me > 0 else np.nan

def classifica_fase(max_fft, gap, copertura, ampiezza, rumore):
    """Classificazione di [F] sec.2, con un controllo di validità in più.

    [F] applica la misura a testi in cui le parole hanno quasi sempre un vettore.
    Alle temperature alte i modelli qui analizzati producono sequenze
    multiscript in cui la copertura di GloVe crolla e l'ACF diventa
    indistinguibile dal rumore: in quel regime il rapporto GAPELMAPER si
    calcola ancora, ma su un segnale che non c'è. Il caso viene marcato come non
    valutabile invece di essere classificato, coerentemente con [F] sec.2.6, dove
    GAPELMAPER risulta non calcolabile alle temperature estreme."""
    if not np.isfinite(max_fft):
        return 'n.d.'
    if copertura < CFG.COPERTURA_MIN or not np.isfinite(ampiezza) or ampiezza < rumore:
        return 'non valutabile'
    if max_fft > CFG.SOGLIA_FFT:
        return 'solido (periodico)'
    if np.isfinite(gap) and gap < 1.0:
        return 'critico (power law)'
    return 'gas (amorfo)'

print("4.6 Fasi: caricate.")

### 4.7 Perché la baseline umana va misurata alla stessa lunghezza

$R$, $\beta$ e il MAPE non sono invarianti di scala: dipendono da quanti token
si osservano. I valori pubblicati in [Z] ($R = 0.17$, $\beta = 0.801$) sono
calcolati su opere letterarie intere, lunghe centinaia di migliaia di token,
mentre i testi generati qui analizzati ne hanno 24 000.

La tabella seguente misura la baseline umana a lunghezze crescenti. Se la deriva
è marcata, confrontare i testi generati con i valori pubblicati sarebbe un
errore sistematico, e la baseline corretta è quella alla lunghezza di lavoro.

In [ ]:
righe = []
ids_umano = ENC.encode(UMANO_TESTO[:3_000_000], disallowed_special=())
for N in (5000, 10000, CFG.N_TOK, 50000, 100000, 300000):
    if N > len(ids_umano):
        continue
    tk = [str(i) for i in ids_umano[:N]]
    tx = ENC.decode(ids_umano[:N])
    a_c, mape = zipf_cumulativo_fit(tk)
    righe.append(dict(N_token=N, alpha=a_c, MAPE=mape, beta=heaps_beta(tk),
                      R=descrittore_R(tk), alpha_DFA=dfa_alpha(serie_rango_caratteri(tx)),
                      TTR=ttr(tk)))
DLUN = pd.DataFrame(righe)
DLUN.to_csv(CFG.OUT / 'baseline_umana_vs_lunghezza.csv', index=False)
display(DLUN.round(4))

fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, col, lab, chiave in zip(axes, ['alpha', 'beta', 'R', 'MAPE'],
                                [r'$\alpha$', r'$\beta$', '$R$', 'MAPE'],
                                ['alpha', 'beta', 'R', 'mape']):
    ax.plot(DLUN.N_token, DLUN[col], 'o-', color=COLORE_UMANO, lw=1.7, ms=6)
    banda_riferimento(ax, chiave, '[Z], opere intere')
    ax.axvline(CFG.N_TOK, color='crimson', ls=':', lw=1.4)
    ax.set_xscale('log'); ax.set_xlabel('token analizzati $N$')
    ax.set_ylabel(lab); ax.set_title(f'{lab} — War and Peace', fontsize=10)
    ax.legend(fontsize=7)
plt.suptitle('La baseline umana dipende dalla lunghezza: la riga rossa è la '
             'lunghezza di lavoro di questo notebook', fontsize=11)
plt.tight_layout(); salva('01_baseline_vs_lunghezza')

print(f"Alla lunghezza di lavoro N = {CFG.N_TOK:,} token, la baseline umana vale:")
_r = DLUN[DLUN.N_token == CFG.N_TOK].iloc[0]
for c in ('alpha', 'beta', 'R', 'MAPE'):
    pub = CFG.RIF_Z.get({'alpha': 'alpha', 'beta': 'beta', 'R': 'R', 'MAPE': 'mape'}[c])
    print(f"  {c:>6} = {_r[c]:.3f}   (pubblicato su opere intere: {pub[0]} ± {pub[1]})")

## 5. Calcolo delle metriche

Tutte le misure per documento vengono calcolate una sola volta e raccolte in un
unico `DataFrame`, che viene salvato. Le sezioni successive si limitano a
leggerlo.

In [ ]:
def analizza_documento(tok, testo, par, rng):
    """Tutte le metriche per singolo documento, con l'indicazione del paper."""
    out = dict(n_token=len(tok), n_tipi=len(set(tok)), n_char=len(testo),
               n_parole=len(par))
    if len(tok) < CFG.MIN_TOK:
        out['valido'] = False
        return out
    out['valido'] = True

    # --- [Z] Zipf --------------------------------------------------------
    a_c, mape = zipf_cumulativo_fit(tok)
    out['alpha'] = a_c
    out['MAPE'] = mape
    out['alpha_rango'] = zipf_alpha_rango(tok)

    # --- [Z] Heaps e R ---------------------------------------------------
    out['beta'] = heaps_beta(tok)
    out['R'] = descrittore_R(tok)
    out['TTR'] = ttr(tok)
    out['TTR_parole'] = ttr(par)
    out['rip_8gram'] = ngram_ripetuti(par)
    rm, rs = distanze_ripetizione(par)
    out['dist_rip_media'], out['dist_rip_sd'] = rm, rs

    # --- [L] DFA sui caratteri -------------------------------------------
    serie = serie_rango_caratteri(testo)
    out['alpha_DFA'] = dfa_alpha(serie)
    out['alpha_DFA_mescolato'] = dfa_alpha_mescolato(serie, rng=rng)

    # --- [C] compressione -------------------------------------------------
    out['R_gzip'] = gzip_ratio(testo)
    out['compr_condizionata'] = compressione_condizionata(testo)
    pm, pt, pv = statistiche_prefissi(testo)
    out['prefix_media'], out['prefix_trend'], out['prefix_var'] = pm, pt, pv
    gap, d_ncd = contributo_ordine(testo, rng)
    out['gap_shuffle'], out['NCD_shuffle'] = gap, d_ncd
    out['H_char_norm'] = entropia_normalizzata(testo)
    out['H_parola_norm'] = entropia_normalizzata(par)
    out['quota_non_ascii'] = sum(1 for c in testo if ord(c) > 127) / len(testo)

    # --- [L] entropia, divergenza dall'umano, plagio ----------------------
    out['H_bit_char'] = tasso_entropia(testo[:2 * CFG.KL_CHUNK])
    out['D_s_umano'] = kl_simmetrica(testo[:CFG.KL_CHUNK], UMANO_TXT[:CFG.KL_CHUNK])
    out['LCS_umano'] = lcs(testo[:CFG.LCS_CHUNK], UMANO_TESTO[:CFG.LCS_RIF_CHUNK])
    return out

t0 = time.time()
righe = []
for i, r in enumerate(DOCS):
    rng = np.random.default_rng(CFG.SEED + i)
    base = dict(modello=r['_modello'], temperatura=r['temperature'],
                sample_id=r.get('sample_id', i),
                n_continuations=r.get('n_continuations'))
    righe.append({**base, **analizza_documento(r['_tok'], r['_testo'], r['_parole'], rng)})
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(DOCS)}  ({time.time()-t0:.0f}s)", end='\r')

DF = pd.DataFrame(righe)
DF = DF[DF.valido].drop(columns=['valido'])
DF.to_csv(CFG.OUT / 'metriche_per_documento.csv', index=False)
print(f"\nCalcolate {len(DF)} righe in {time.time()-t0:.0f}s")

# --- baseline umana, stesse funzioni, stessa lunghezza -----------------------
UMANO = analizza_documento(UMANO_TOK, UMANO_TXT, UMANO_PAROLE,
                           np.random.default_rng(CFG.SEED))
UMANO['D_s_umano'] = 0.0
# L'LCS del testo umano contro sé stesso è un auto-confronto e vale banalmente
# tutta la finestra: non è un valore di riferimento e va tolto.
UMANO['LCS_umano'] = np.nan
pd.Series(UMANO).to_csv(CFG.OUT / 'baseline_umana.csv')
print(f"\nBaseline umana (War and Peace, continuazione, {UMANO['n_token']:,} token):")
display(pd.DataFrame([{k: (round(v, 4) if isinstance(v, float) else v)
                       for k, v in UMANO.items() if k != 'valido'}]).T.rename(columns={0: 'valore'}))

### 5.1 Funzioni di disegno

In [ ]:
def plot_vs_T(df, col, ylabel, titolo, ax=None, umano=None, chiave_rif=None,
              logy=False, estremo=None, legenda=True, modelli=None):
    """Una curva per modello in funzione di T, con barre d'errore dove c'è più
    di un campione, la baseline umana in nero e, se disponibile, la banda
    pubblicata in [Z]."""
    ax = ax or plt.gca()
    modelli = modelli or [m for m in MODELLI if m in set(df.modello)]
    if chiave_rif:
        banda_riferimento(ax, chiave_rif, '[Z], opere intere')
    for m in modelli:
        d = df[df.modello == m]
        if col not in d.columns or d[col].notna().sum() == 0:
            continue
        g = d.groupby('temperatura')[col]
        mu, sd, n = g.mean(), g.std(), g.count()
        sd = sd.where(n > 1, 0.0).fillna(0.0)
        st = stile_modello(m)
        ax.errorbar(mu.index, mu.values, yerr=sd.values, capsize=3, ms=5,
                    lw=1.6, label=m, **st)
        if estremo in ('min', 'max') and mu.notna().any():
            Tc = mu.idxmin() if estremo == 'min' else mu.idxmax()
            ax.axvline(Tc, ls=':', lw=1, alpha=.45, color=st['color'])
    if umano is not None and np.isfinite(umano):
        ax.axhline(umano, color=COLORE_UMANO, lw=1.9, ls='--',
                   label=f'umano, {CFG.N_TOK//1000}k token')
    ax.set_xlabel('temperatura $T$'); ax.set_ylabel(ylabel)
    ax.set_title(titolo, fontsize=10)
    if logy:
        ax.set_yscale('log')
    if legenda:
        ax.legend(fontsize=6.5)
    return ax

def plot_curve_per_T(ax, dati, xlabel, ylabel, titolo, loglog=True, ncol=2, umano=None):
    """Una curva per temperatura, con la palette qualitativa."""
    for T in sorted(dati):
        x, y = dati[T]
        if len(x) == 0:
            continue
        st = stile_T(T, TEMPERATURE)
        f = ax.loglog if loglog else ax.plot
        f(x, y, lw=1.3, ms=3.5, markevery=max(1, len(x) // 12), label=f'$T$={T:g}', **st)
    if umano is not None:
        x, y = umano
        (ax.loglog if loglog else ax.plot)(x, y, color=COLORE_UMANO, lw=2.4,
                                           ls='--', label='umano', zorder=10)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(titolo, fontsize=10)
    ax.legend(fontsize=6.5, ncol=ncol)

def esempi_per_T(modello):
    """Un documento rappresentativo per ciascuna temperatura."""
    out = {}
    for T in TEMPERATURE:
        c = [r for r in DOCS if r['_modello'] == modello and r['temperature'] == T
             and r['_valido']]
        if c:
            out[T] = c[0]
    return out

print("Funzioni di disegno pronte.")

### 5.2 I testi ricuciti introducono artefatti?

I dataset sono stati generati in modalità `continuation`: dopo ogni token EOS la
generazione riparte dal testo accumulato, finché non si raggiungono i 24 000
token. Un modello autoregressivo è privo di stato, quindi riprendere dal testo
accumulato produce la stessa distribuzione; ma le riprese si addensano dove la
probabilità di EOS è massima, e vale la pena verificare che i punti di giunzione
non producano discontinuità proprio nelle misure sensibili all'ordine.

Il controllo: le metriche d'ordine correlano con il numero di riprese?

In [ ]:
d = DF.dropna(subset=['n_continuations'])
if len(d):
    print(f"Documenti con il campo n_continuations: {len(d)}/{len(DF)}")
    print(f"Riprese: mediana {d.n_continuations.median():.0f}, "
          f"massimo {d.n_continuations.max():.0f}, "
          f"quota di testi ricuciti {(d.n_continuations > 0).mean():.0%}\n")
    display(d.pivot_table(index='temperatura', columns='modello', values='n_continuations',
                          aggfunc='median').fillna(0).astype(int))

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.1))
    for m in MODELLI:
        dd = d[d.modello == m]
        if not len(dd):
            continue
        g = dd.groupby('temperatura')['n_continuations'].median()
        axes[0].plot(g.index, g.values, ms=5, lw=1.6, label=m, **stile_modello(m))
    axes[0].set_xlabel('temperatura $T$'); axes[0].set_ylabel('riprese (mediana)')
    axes[0].set_title('Riprese per temperatura\n(seguono il massimo di $p_{EOS}$)', fontsize=10)
    axes[0].legend(fontsize=6.5)

    for ax, col, lab in ((axes[1], 'alpha_DFA', r'$\alpha_{\mathrm{DFA}}$'),
                         (axes[2], 'rip_8gram', '8-grammi ripetuti')):
        for m in MODELLI:
            dd = d[d.modello == m].dropna(subset=[col])
            if not len(dd):
                continue
            st = stile_modello(m)
            ax.scatter(dd.n_continuations, dd[col], s=28, alpha=.7,
                       color=st['color'], marker=st['marker'], label=m)
        ax.set_xlabel('numero di riprese'); ax.set_ylabel(lab)
        dd = d.dropna(subset=[col])
        if HAS_SCIPY and len(dd) > 8 and dd.n_continuations.nunique() > 2:
            rho, pv = _sps.spearmanr(dd.n_continuations, dd[col])
            esito = 'nessun artefatto' if pv > .05 else 'correlato: da approfondire'
            ax.set_title(f'{lab} vs riprese\n' + r'$\rho$=' +
                         f'{rho:+.2f}, p={pv:.3f} — {esito}', fontsize=9.5)
        ax.legend(fontsize=6)
    plt.tight_layout(); salva('02_artefatti_continuazione')

    if HAS_SCIPY:
        print("\nCorrelazione di Spearman fra riprese e metriche sensibili all'ordine:")
        for col in ('alpha_DFA', 'rip_8gram', 'R', 'alpha', 'R_gzip'):
            dd = d.dropna(subset=[col])
            if len(dd) > 8 and dd.n_continuations.nunique() > 2:
                rho, pv = _sps.spearmanr(dd.n_continuations, dd[col])
                print(f"  {col:20s} rho={rho:+.3f}  p={pv:.3f}"
                      f"{'' if pv > .05 else '   <-- significativa'}")
        print("\nNota: le riprese sono a loro volta funzione di T, quindi una")
        print("correlazione qui non implica un artefatto di giunzione. Il numero")
        print("mediano di riprese per temperatura va comunque riportato.")
else:
    print("Nessun campo n_continuations: generazione in una sola chiamata.")

## 6. Legge di Zipf — [Z] §4.3, [L] §V-A

[Z] misura l'aderenza alla legge di Zipf sull'**istogramma cumulativo** e la
quantifica con il MAPE del fit power-law. Il risultato centrale del paper è che
il MAPE ha un **minimo pronunciato** in funzione della temperatura: è lì che il
testo generato è più vicino a una legge di potenza, cioè allo stato critico.
Sopra e sotto quel valore la legge di Zipf non vale nemmeno approssimativamente.

A sinistra gli istogrammi cumulativi, uno per temperatura, nello stile della
Fig. 6 di [Z]; a destra la distribuzione rango-frequenza classica.

In [ ]:
for m in MODELLI:
    ex = esempi_per_T(m)
    if not ex:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
    plot_curve_per_T(axes[0], {T: istogramma_cumulativo(r['_tok']) for T, r in ex.items()},
                     'numero di occorrenze $x$', r'$P(\mathrm{occorrenze} > x)$',
                     f'Istogramma cumulativo — {m}   (cfr. [Z] Fig. 6)',
                     umano=istogramma_cumulativo(UMANO_TOK))
    plot_curve_per_T(axes[1], {T: zipf_rango_frequenza(r['_tok']) for T, r in ex.items()},
                     'rango $r$', 'frequenza $f(r)$',
                     f'Rango-frequenza — {m}   (cfr. [L] Fig. 2A)',
                     umano=zipf_rango_frequenza(UMANO_TOK))
    plt.tight_layout(); salva(f'03_zipf_curve_{m.replace(".", "").replace("-", "_")}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
plot_vs_T(DF, 'MAPE', 'MAPE del fit power-law',
          'Bontà del fit di Zipf\nil minimo individua lo stato critico [Z] §4.3',
          axes[0], UMANO['MAPE'], chiave_rif='mape', estremo='min')
plot_vs_T(DF, 'alpha', r'esponente di Zipf $\alpha$',
          r"Esponente $\alpha$ dall'istogramma cumulativo",
          axes[1], UMANO['alpha'], chiave_rif='alpha')
plt.tight_layout(); salva('04_zipf_vs_T')

print("Temperatura di MAPE minimo (stato critico secondo [Z]):")
for m in MODELLI:
    g = DF[DF.modello == m].groupby('temperatura')['MAPE'].mean().dropna()
    if len(g) >= 3:
        print(f"  {m:16s} T* = {g.idxmin():.1f}   MAPE = {g.min():.3f}"
              f"   (umano alla stessa lunghezza: {UMANO['MAPE']:.3f})")
print(f"\n[Z] riporta, per i modelli Qwen base generati da un singolo token,")
print(f"un minimo del MAPE fra t = 1.0 e t = 1.2.")

## 7. Legge di Heaps e descrittore $R$ — [Z] §4.2, [L] §V-A

[Z] osserva che la curva di crescita del vocabolario dei testi generati è così
lontana da una legge di potenza che fittarne l'esponente spesso non ha senso: a
bassa temperatura il testo degenera e smette di produrre token nuovi, ad alta
temperatura il vocabolario esplode. Per questo introduce $R$, il rapporto fra i
tipi che compaiono per la prima volta nella seconda e nella prima metà del
testo, che è sempre definito.

Nota su cosa aspettarsi: [Z] §4.5 osserva che con un **prompt lungo** — come
qui, 2000 token — la ricchezza lessicale del prompt domina il vocabolario del
testo generato, e questo *sposta i massimi di $R$ verso le temperature alte*
rispetto alla generazione da un singolo token. La curva $R(T)$ va letta con
questa avvertenza.

In [ ]:
for m in MODELLI:
    ex = esempi_per_T(m)
    if not ex:
        continue
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    dati = {}
    for T, r in ex.items():
        n = heaps_curva(r['_tok'])
        idx = np.unique(np.logspace(0, np.log10(len(n)), 80).astype(int))
        idx = idx[idx <= len(n)]
        dati[T] = (idx, n[idx - 1])
    nu = heaps_curva(UMANO_TOK)
    iu = np.unique(np.logspace(0, np.log10(len(nu)), 80).astype(int))
    iu = iu[iu <= len(nu)]
    plot_curve_per_T(ax, dati, 'posizione nel testo $t$ [token]',
                     'tipi distinti $n(t)$',
                     f'Curve di Heaps — {m}   (cfr. [Z] Fig. 3, [L] Fig. 3A)',
                     umano=(iu, nu[iu - 1]))
    plt.tight_layout(); salva(f'05_heaps_{m.replace(".", "").replace("-", "_")}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
plot_vs_T(DF, 'R', 'descrittore $R$',
          r'$R$ = tipi nuovi nella 2ª metà / nella 1ª' + '\n[Z] §4.2',
          axes[0], UMANO['R'], chiave_rif='R', estremo='max')
plot_vs_T(DF, 'beta', r'esponente di Heaps $\beta$',
          'Crescita del vocabolario', axes[1], UMANO['beta'], chiave_rif='beta')
plot_vs_T(DF, 'TTR', 'type-token ratio',
          'Diversità lessicale (token)', axes[2], UMANO['TTR'])
plt.tight_layout(); salva('06_heaps_R_vs_T')

print("Confronto con la baseline umana alla stessa lunghezza "
      f"(R = {UMANO['R']:.3f}, beta = {UMANO['beta']:.3f}):")
for m in MODELLI:
    d = DF[DF.modello == m]
    gR = d.groupby('temperatura')['R'].mean().dropna()
    gB = d.groupby('temperatura')['beta'].mean().dropna()
    if len(gR) >= 3:
        print(f"  {m:16s} R più vicino all'umano a T = {(gR - UMANO['R']).abs().idxmin():.1f}"
              f"   |   beta più vicino a T = {(gB - UMANO['beta']).abs().idxmin():.1f}"
              f"   |   massimo di R a T = {gR.idxmax():.1f}")

## 8. Correlazioni a lungo raggio — DFA, [L] §III-B e §V-B

La DFA è la misura con cui [L] verifica se il testo artificiale possiede le
correlazioni a lungo raggio del linguaggio naturale. Sul testo di Dickens
originale l'esponente è nettamente sopra $0.5$; sul testo mescolato scende a
$0.5$, che è il valore di una sequenza non correlata.

Attenzione a leggere il caso degenere: quando il modello entra in un ciclo di
ripetizione la serie diventa quasi periodica e la fluttuazione **satura**, cioè
smette di crescere con la scala. L'esponente crolla verso zero. Un $\alpha_{\mathrm{DFA}}$
molto basso qui non indica assenza di struttura, ma struttura periodica: è la
fase solida di [F], e va letta insieme alla quota di $n$-grammi ripetuti.

In [ ]:
for m in MODELLI:
    ex = esempi_per_T(m)
    if not ex:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.5))
    dati = {}
    for T, r in ex.items():
        L, F = dfa(serie_rango_caratteri(r['_testo']))
        if len(L):
            dati[T] = (L, F / F[0])
    Lu, Fu = dfa(serie_rango_caratteri(UMANO_TXT))
    plot_curve_per_T(axes[0], dati, 'scala $L$ [caratteri]', '$F(L)$ normalizzata',
                     f'DFA sui caratteri — {m}   (cfr. [L] Fig. 4A)',
                     umano=(Lu, Fu / Fu[0]) if len(Lu) else None)
    sref = np.array([CFG.DFA_LI, CFG.DFA_LI * 30], dtype=float)
    axes[0].loglog(sref, (sref / sref[0]) ** 0.5, 'k:', lw=1.5,
                   label=r'$\alpha_{\mathrm{DFA}}=0.5$')
    axes[0].legend(fontsize=6.5, ncol=2)

    d = DF[DF.modello == m]
    g = d.groupby('temperatura')
    mu, sd, n = g['alpha_DFA'].mean(), g['alpha_DFA'].std(), g['alpha_DFA'].count()
    st = stile_modello(m)
    axes[1].errorbar(mu.index, mu.values, yerr=sd.where(n > 1, 0).fillna(0).values,
                     capsize=3, ms=5, lw=1.6, label='originale', **st)
    axes[1].plot(g['alpha_DFA_mescolato'].mean().index,
                 g['alpha_DFA_mescolato'].mean().values, 's--', ms=4,
                 color='grey', label='mescolato (null model)')
    axes[1].axhline(0.5, color='k', ls=':', lw=1.2)
    axes[1].axhline(UMANO['alpha_DFA'], color=COLORE_UMANO, ls='--', lw=1.9,
                    label='umano')
    axes[1].set_xlabel('temperatura $T$')
    axes[1].set_ylabel(r'$\alpha_{\mathrm{DFA}}$')
    axes[1].set_title(f'Esponente DFA — {m}   (cfr. [L] Fig. 4B)', fontsize=10)
    axes[1].legend(fontsize=7.5)
    plt.tight_layout(); salva(f'07_dfa_{m.replace(".", "").replace("-", "_")}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
plot_vs_T(DF, 'alpha_DFA', r'$\alpha_{\mathrm{DFA}}$',
          'Correlazioni a lungo raggio — tutti i modelli', axes[0],
          UMANO['alpha_DFA'], estremo='max')
axes[0].axhline(0.5, color='grey', ls=':', lw=1.2)
axes[0].text(axes[0].get_xlim()[0], 0.505, ' non correlato', fontsize=7,
             color='grey', va='bottom')

# la degenerazione produce alpha bassi: mostriamo il legame esplicito
for m in MODELLI:
    d = DF[DF.modello == m]
    st = stile_modello(m)
    axes[1].scatter(d.rip_8gram, d.alpha_DFA, s=30, alpha=.75,
                    color=st['color'], marker=st['marker'], label=m)
axes[1].scatter([UMANO['rip_8gram']], [UMANO['alpha_DFA']], s=190, marker='*',
                color=COLORE_UMANO, zorder=6, label='umano')
axes[1].axhline(0.5, color='grey', ls=':', lw=1.2)
axes[1].set_xlabel('quota di 8-grammi ripetuti')
axes[1].set_ylabel(r'$\alpha_{\mathrm{DFA}}$')
axes[1].set_title('Il ramo a sinistra è la fase periodica:\nripetizione alta, '
                  'fluttuazione satura', fontsize=10)
axes[1].legend(fontsize=6.5)
plt.tight_layout(); salva('08_dfa_tutti')

print("Verifica del null model: l'esponente del testo mescolato deve stare su 0.5")
g = DF.groupby('modello')['alpha_DFA_mescolato'].agg(['mean', 'std', 'min', 'max'])
display(g.round(3))
print(f"\nUmano mescolato: {UMANO['alpha_DFA_mescolato']:.3f}   "
      f"umano originale: {UMANO['alpha_DFA']:.3f}")

### 8.1 Robustezza rispetto all'ordine del detrending

La DFA di ordine $n$ rimuove dalle finestre le tendenze polinomiali fino al grado
$n$: tutte le misure della sezione precedente usano l'ordine 1, cioè eliminano le
tendenze lineari locali. Ripetere la stima con l'ordine 2 verifica che
$\alpha_{\mathrm{DFA}}$ non sia gonfiato da tendenze di grado superiore rimaste
nel segnale, ed è il controllo standard raccomandato in letteratura per
distinguere le correlazioni a lungo raggio dalle non stazionarietà.

Il confronto va letto insieme alla quota di $n$-grammi ripetuti: dove il testo
degenera in cicli, la serie dei ranghi non è stazionaria e i due ordini possono
divergere.

In [ ]:
# =====================================================================
# 8.1 Robustezza rispetto all'ordine del detrending
# =====================================================================
t0 = time.time()
righe_o2 = []
for i, r in enumerate(DOCS):
    if not r['_valido']:
        continue
    righe_o2.append(dict(modello=r['_modello'], temperatura=r['temperature'],
                         sample_id=r.get('sample_id', i),
                         alpha_DFA_o2=dfa_alpha(serie_rango_caratteri(r['_testo']),
                                                ordine=2)))
DO2 = pd.DataFrame(righe_o2)
DF = DF.merge(DO2, on=['modello', 'temperatura', 'sample_id'], how='left')
DF['scarto_ordine'] = DF.alpha_DFA_o2 - DF.alpha_DFA
UMANO['alpha_DFA_o2'] = dfa_alpha(serie_rango_caratteri(UMANO_TXT), ordine=2)
DF.to_csv(CFG.OUT / 'metriche_per_documento.csv', index=False)
print(f"Stime di ordine 2 calcolate per {len(DO2)} documenti in {time.time()-t0:.0f}s")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))

# --- (a) i due esponenti in funzione della temperatura ----------------------
ax = axes[0]
for col, lab, col_linea, ls in ((r'alpha_DFA', 'ordine 1 (lineare)', '#2F4E7E', '-'),
                                ('alpha_DFA_o2', 'ordine 2 (quadratico)', '#C1502E', '--')):
    g = DF.groupby('temperatura')[col]
    mu, sd = g.mean(), g.std().fillna(0)
    ax.plot(mu.index, mu.values, ls=ls, marker='o', ms=5, lw=1.7,
            color=col_linea, label=lab)
    ax.fill_between(mu.index, mu - sd, mu + sd, color=col_linea, alpha=.12)
ax.axhline(UMANO['alpha_DFA'], color='#2F4E7E', ls=':', lw=1.4)
ax.axhline(UMANO['alpha_DFA_o2'], color='#C1502E', ls=':', lw=1.4)
ax.axhline(0.5, color='grey', ls=':', lw=1)
ax.set_xlabel('temperatura $T$'); ax.set_ylabel(r'$\alpha_{\mathrm{DFA}}$')
ax.set_title('Esponente ai due ordini di detrending\n(le linee punteggiate sono '
             'la baseline umana)', fontsize=10)
ax.legend(fontsize=7.5)

# --- (b) confronto documento per documento ---------------------------------
ax = axes[1]
for m in MODELLI:
    d = DF[DF.modello == m]
    st = stile_modello(m)
    ax.scatter(d.alpha_DFA, d.alpha_DFA_o2, s=26, alpha=.7,
               color=st['color'], marker=st['marker'], label=m)
lim = [min(DF.alpha_DFA.min(), DF.alpha_DFA_o2.min()) - .05,
       max(DF.alpha_DFA.max(), DF.alpha_DFA_o2.max()) + .05]
ax.plot(lim, lim, 'k--', lw=1.2, label='identità')
ax.scatter([UMANO['alpha_DFA']], [UMANO['alpha_DFA_o2']], s=190, marker='*',
           color=COLORE_UMANO, zorder=6, label='umano')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(r'$\alpha_{\mathrm{DFA}}$, ordine 1')
ax.set_ylabel(r'$\alpha_{\mathrm{DFA}}$, ordine 2')
ax.set_title('Confronto documento per documento', fontsize=10)
ax.legend(fontsize=6.5)

# --- (c) lo scarto segue la degenerazione ----------------------------------
ax = axes[2]
for m in MODELLI:
    d = DF[DF.modello == m]
    st = stile_modello(m)
    ax.scatter(d.rip_8gram, d.scarto_ordine.abs(), s=26, alpha=.7,
               color=st['color'], marker=st['marker'], label=m)
ax.scatter([UMANO['rip_8gram']], [abs(UMANO['alpha_DFA_o2'] - UMANO['alpha_DFA'])],
           s=190, marker='*', color=COLORE_UMANO, zorder=6, label='umano')
ax.set_xlabel('quota di 8-grammi ripetuti')
ax.set_ylabel(r'$|\alpha_{o2} - \alpha_{o1}|$')
ax.set_title('Lo scarto fra i due ordini\nsegue la degenerazione del testo', fontsize=10)
ax.legend(fontsize=6.5)

plt.tight_layout(); salva('08b_robustezza_ordine_dfa')

# --- tabella e sintesi -------------------------------------------------------
TAB_O2 = DF.groupby('temperatura')[['alpha_DFA', 'alpha_DFA_o2', 'scarto_ordine']].mean().round(4)
TAB_O2.to_csv(CFG.OUT / 'robustezza_ordine_dfa.csv')
display(TAB_O2)

sc = DF.scarto_ordine.dropna()
print(f"Scarto (ordine 2 meno ordine 1) sui {len(sc)} documenti:")
print(f"  media           {sc.mean():+.4f}")
print(f"  |scarto| mediano {sc.abs().median():.4f}   massimo {sc.abs().max():.4f}")
print(f"  correlazione fra i due esponenti: {DF.alpha_DFA.corr(DF.alpha_DFA_o2):.4f}")
print(f"\nBaseline umana: ordine 1 = {UMANO['alpha_DFA']:.4f}, "
      f"ordine 2 = {UMANO['alpha_DFA_o2']:.4f}, "
      f"scarto {UMANO['alpha_DFA_o2'] - UMANO['alpha_DFA']:+.4f}")

if HAS_SCIPY:
    d = DF[['rip_8gram', 'scarto_ordine']].dropna()
    rho, pv = _sps.spearmanr(d.rip_8gram, d.scarto_ordine.abs())
    print(f"\nSpearman(8-grammi ripetuti, |scarto|) = {rho:+.3f}  (p = {pv:.1e}, n = {len(d)})")

_alta = DF[DF.temperatura >= 1.2].scarto_ordine.abs().median()
_bassa = DF[DF.temperatura.between(0.5, 0.8)].scarto_ordine.abs().median()
print(f"\n|scarto| mediano a T fra 0.5 e 0.8 : {_bassa:.4f}")
print(f"|scarto| mediano a T >= 1.2        : {_alta:.4f}")
print("\nLettura: dove il testo e' stazionario - baseline umana e regime disordinato -")
print("i due ordini coincidono, e l'esponente non dipende dalla scelta del detrending.")
print("Nel regime degenere le due stime divergono, perche' la serie dei ranghi non e'")
print("stazionaria: la' l'esponente di ordine 1 va letto come indicatore di")
print("periodicita' e non come misura di correlazione a lungo raggio.")



## 9. Entropia, divergenza dal testo umano, plagio — [L] §III-C e §III-D

[L] stima entropia e divergenza KL con algoritmi di compressione, e trova per
gli LSTM un **minimo netto della KL dal corpus originale attorno a $T \approx 1$**:
è la temperatura alla quale la struttura del testo generato è più vicina a
quella del testo umano. È il criterio più diretto di tutto il notebook, perché
non dipende da soglie teoriche ma dal confronto con un testo reale.

La LCS misura invece quanta parte del testo è copiata letteralmente dal
riferimento. Poiché il prompt viene da *War and Peace*, una sottostringa lunga
in comune indica che il modello sta riproducendo a memoria il romanzo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
plot_vs_T(DF, 'D_s_umano', r'$D_s$ [bit/carattere]',
          'Divergenza KL simmetrizzata dal testo umano\nil minimo dà $T^*$ '
          '(cfr. [L] Fig. 5A)', axes[0], estremo='min')
axes[0].axhline(0, color='k', lw=1)
plot_vs_T(DF, 'H_bit_char', 'entropia [bit/carattere]',
          'Entropia stimata per compressione\n(cfr. [L] Fig. 5B)',
          axes[1], UMANO['H_bit_char'])
plot_vs_T(DF, 'LCS_umano', 'LCS [caratteri]',
          'Sottostringa più lunga in comune con Tolstoj\n(cfr. [L] §III-D)',
          axes[2], estremo='max')
plt.tight_layout(); salva('09_entropia_kl_lcs')

print("Temperatura che minimizza la divergenza dal testo umano:")
righe = []
for m in MODELLI:
    g = DF[DF.modello == m].groupby('temperatura')['D_s_umano'].mean().dropna()
    if len(g) >= 3:
        print(f"  {m:16s} T* = {g.idxmin():.1f}   D_s = {g.min():.3f} bit/carattere")
        righe.append(dict(modello=m, T_star=g.idxmin(), D_s=g.min()))
n_neg = (DF.D_s_umano < -1e-9).sum()
print(f"\nDivergenze negative: {n_neg} su {DF.D_s_umano.notna().sum()} "
      f"({'stimatore coerente' if n_neg == 0 else 'da controllare'})")
print(f"Entropia del testo umano alla stessa lunghezza: {UMANO['H_bit_char']:.3f} bit/carattere")
print(f"LCS umano-vs-sé stesso non è definita; LCS massima osservata sui generati: "
      f"{DF.LCS_umano.max():.0f} caratteri")

## 10. Struttura misurata per compressione — [C] §3.2 e §4.1.1

[C] mostra che la compressione lossless è un rivelatore semplice e
model-agnostic della regolarità strutturale: testi più regolari si comprimono
meglio. La Fig. 1A di [C] lega il compression ratio all'entropia del vocabolario.

Uno sweep di temperatura è, di fatto, uno **sweep controllato di entropia**: la
temperatura riscala i logit prima del softmax e quindi allarga o concentra la
distribuzione da cui i token vengono estratti. Il pannello in basso a destra
riproduce l'asse della Fig. 1A di [C] usando la temperatura come variabile di
controllo, e colloca su di esso il testo umano.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8.6))
plot_vs_T(DF, 'R_gzip', r'$R_{\mathrm{gzip}} = C(x)/|x|$',
          'Compression ratio [C] eq.(4)', axes[0, 0], UMANO['R_gzip'])
plot_vs_T(DF, 'compr_condizionata', 'compressione condizionata',
          'Prevedibilità dal contesto [C]', axes[0, 1], UMANO['compr_condizionata'])
plot_vs_T(DF, 'prefix_trend', 'pendenza della curva di prefisso',
          'Prefix ratio trend [C] Fig.1C', axes[0, 2], UMANO['prefix_trend'])
plot_vs_T(DF, 'NCD_shuffle', 'NCD(originale, permutato)',
          'Contributo dell\'ordine delle parole [C]', axes[1, 0], UMANO['NCD_shuffle'])
plot_vs_T(DF, 'dist_rip_media', 'distanza media fra ripetizioni',
          'Struttura delle ripetizioni [C]', axes[1, 1], UMANO['dist_rip_media'], logy=True)

ax = axes[1, 2]
for m in MODELLI:
    d = DF[DF.modello == m]
    st = stile_modello(m)
    ax.scatter(d.H_parola_norm, d.R_gzip, s=30, alpha=.75,
               color=st['color'], marker=st['marker'], label=m)
ax.scatter([UMANO['H_parola_norm']], [UMANO['R_gzip']], s=200, marker='*',
           color=COLORE_UMANO, zorder=6, label='umano')
ax.set_xlabel('entropia normalizzata a livello di parola')
ax.set_ylabel(r'$R_{\mathrm{gzip}}$')
ax.set_title('Entropia del vocabolario vs comprimibilità\n(asse della Fig. 1A di [C], '
             'percorso lungo $T$)', fontsize=10)
ax.legend(fontsize=6.5)
plt.tight_layout(); salva('10_compressione')

# --- curve di prefisso, replica della Fig. 1C di [C] -------------------------
fig, axes = plt.subplots(1, len(MODELLI), figsize=(4.4 * len(MODELLI), 4.0), squeeze=False)
for j, m in enumerate(MODELLI):
    ax = axes[0, j]
    ex = esempi_per_T(m)
    for T, r in ex.items():
        x, y = curva_prefissi(r['_testo'])
        if len(x):
            ax.plot(x, y, lw=1.2, ms=3, markevery=4, label=f'$T$={T:g}',
                    **stile_T(T, TEMPERATURE))
    xu, yu = curva_prefissi(UMANO_TXT)
    if len(xu):
        ax.plot(xu, yu, color=COLORE_UMANO, lw=2.4, ls='--', label='umano', zorder=10)
    ax.set_xlabel('numero di frasi'); ax.set_title(m, fontsize=10)
    if j == 0:
        ax.set_ylabel(r'$R_{\mathrm{gzip}}$ del prefisso')
    if j == len(MODELLI) - 1:
        ax.legend(fontsize=6, ncol=2, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.suptitle('Curve di prefisso: come la regolarità si accumula con la lunghezza '
             '(cfr. [C] Fig. 1C)', fontsize=11)
plt.tight_layout(); salva('11_curve_prefissi')

print("Nota metodologica: le curve di prefisso sono definite su unità-frase. Alle")
print("temperature alte la punteggiatura si degrada e la segmentazione in frasi")
print("diventa arbitraria; il numero di frasi rilevate per temperatura è:")
_fr = pd.DataFrame([dict(modello=r['_modello'], temperatura=r['temperature'],
                         n_frasi=len(dividi_frasi(r['_testo'])))
                    for r in DOCS if r['_valido']])
display(_fr.pivot_table(index='temperatura', columns='modello', values='n_frasi',
                        aggfunc='median').astype(int))
print(f"Testo umano, stessa lunghezza: {len(dividi_frasi(UMANO_TXT))} frasi")

## 11. Le tre fasi del testo generato — [F]

[F] classifica i testi generati in tre fasi, per analogia con gli stati della
materia, guardando l'autocorrelazione della traiettoria di embedding:

* **solido (periodico)** — a bassa temperatura il testo degenera in cicli; l'ACF
  è periodica e la sua trasformata di Fourier ha picchi netti;
* **critico** — l'ACF decade a legge di potenza: è lo stato che [F] associa alla
  presenza di struttura gerarchica, e che [F] colloca fra $T = 0.7$ e $T = 1.0$;
* **gas (amorfo)** — ad alta temperatura le correlazioni decadono
  esponenzialmente e non c'è struttura a lungo raggio.

Il parametro d'ordine della prima transizione è $\max\lvert\mathrm{FFT}\rvert$
dell'ACF; la seconda si distingue con il GAPELMAPER, rapporto fra l'errore del
fit a legge di potenza e quello del fit esponenziale.

**Un avvertimento sulla validità della misura.** [F] usa embedding GloVe
multilingue su testi in cui quasi tutte le parole hanno un vettore. I modelli
qui analizzati, alle temperature alte, producono sequenze multiscript in cui la
copertura di GloVe crolla e l'ampiezza dell'ACF scende sotto il livello di
rumore: il GAPELMAPER resta calcolabile ma si applica a un segnale che non c'è.
Quei casi vengono marcati come **non valutabili** invece che classificati.
Lo stesso [F] §2.6 segnala che GAPELMAPER non è calcolabile alle temperature
estreme del suo sweep.

In [ ]:
# --- caricamento di GloVe (scaricato una volta sola nella cache) -------------
GLOVE_URL = ('https://github.com/RaRe-Technologies/gensim-data/releases/download/'
             'glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz')
GLOVE_FILE = CFG.CACHE / 'glove-wiki-gigaword-100.gz'

if not GLOVE_FILE.exists():
    print(f"Scarico GloVe (circa 134 MB) in {GLOVE_FILE} ...")
    import urllib.request
    urllib.request.urlretrieve(GLOVE_URL, GLOVE_FILE)
    print("  fatto.")

t0 = time.time()
EMB = {}
with gzip.open(GLOVE_FILE, 'rt', encoding='utf-8') as f:
    intestazione = f.readline().split()
    for line in f:
        p = line.rstrip().split(' ')
        EMB[p[0]] = np.asarray(p[1:], dtype=np.float32)
EMB_DIM = len(next(iter(EMB.values())))
print(f"GloVe: {len(EMB):,} parole, dimensione {EMB_DIM}, caricato in {time.time()-t0:.0f}s")
print("Nota: [F] usa GloVe multilingue (Ferreira et al. 2016); qui si usa la")
print("versione inglese wiki-gigaword a 100 dimensioni, adeguata a un corpus")
print("inglese. La conseguenza è discussa nella cella successiva.")

In [ ]:
LAG_LUNGHI = np.unique(np.concatenate([
    np.arange(1, 51),
    np.logspace(np.log10(50), np.log10(CFG.ACF_LONG_MAX), 60).astype(int)]))

def analizza_fasi(par, rng):
    """ACF, parametro di periodicità, GAPELMAPER e livello di rumore.
    Il livello di rumore è stimato mescolando le parole: qualunque struttura
    residua dell'ACF sotto quella soglia non è distinguibile dal caso."""
    M, noti = traiettoria(par, EMB, EMB_DIM)
    if len(M) < 500 or not noti.any():
        return None
    ac = acf_coseno(M, CFG.ACF_SHORT_LAGS)
    al = acf_coseno(M, LAG_LUNGHI)
    idx = rng.permutation(len(M))
    ac_null = acf_coseno(M[idx], CFG.ACF_SHORT_LAGS)
    rumore = float(np.nanmax(np.abs(ac_null)))
    ampiezza = float(np.nanmax(np.abs(ac)))
    mm = LAG_LUNGHI <= CFG.GAPELMAPER_W
    out = dict(copertura=float(noti.mean()),
               C_lag1=float(ac[0]),
               ampiezza=ampiezza, rumore=rumore,
               max_fft=parametro_periodicita(ac),
               gapelmaper=gapelmaper(LAG_LUNGHI[mm], al[mm]))
    out['fase'] = classifica_fase(out['max_fft'], out['gapelmaper'],
                                  out['copertura'], ampiezza, rumore)
    return out, ac, al

t0 = time.time(); righe = []; ACF_STORE = {}
for i, r in enumerate(DOCS):
    if not r['_valido']:
        continue
    res = analizza_fasi(r['_parole'], np.random.default_rng(CFG.SEED + i))
    if res is None:
        continue
    out, ac, al = res
    chiave = (r['_modello'], r['temperature'], r.get('sample_id', i))
    ACF_STORE[chiave] = (ac, al)
    righe.append(dict(modello=r['_modello'], temperatura=r['temperature'],
                      sample_id=r.get('sample_id', i), **out))
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(DOCS)}  ({time.time()-t0:.0f}s)", end='\r')

DACF = pd.DataFrame(righe)
DACF.to_csv(CFG.OUT / 'fasi_acf.csv', index=False)
print(f"\nACF calcolate per {len(DACF)} documenti in {time.time()-t0:.0f}s")

res_u = analizza_fasi(UMANO_PAROLE, np.random.default_rng(CFG.SEED))
UMANO_ACF = res_u[0] if res_u else {}
if UMANO_ACF:
    print("Umano: " + "  ".join(f"{k}={v:.3f}" for k, v in UMANO_ACF.items()
                                if isinstance(v, float)) + f"  fase={UMANO_ACF['fase']}")

display(DACF.groupby(['modello', 'temperatura'])[['copertura', 'max_fft', 'gapelmaper']]
        .mean().round(3).unstack(0))

In [ ]:
# --- ACF e sua trasformata, un pannello per modello --------------------------
for m in MODELLI:
    chiavi = {}
    for T in TEMPERATURE:
        k = [k for k in ACF_STORE if k[0] == m and k[1] == T]
        if k:
            chiavi[T] = k[0]
    if not chiavi:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3))
    for T, k in chiavi.items():
        axes[0].plot(CFG.ACF_SHORT_LAGS, ACF_STORE[k][0], lw=1.2, ms=3,
                     markevery=12, label=f'$T$={T:g}', **stile_T(T, TEMPERATURE))
    if res_u:
        axes[0].plot(CFG.ACF_SHORT_LAGS, res_u[1], color=COLORE_UMANO, lw=2.3,
                     ls='--', label='umano')
    axes[0].axhline(0, color='k', lw=.8)
    axes[0].set_xlabel(r'lag $\tau$ [parole]'); axes[0].set_ylabel(r'$C(\tau)$')
    axes[0].set_title(f'ACF a lag corti — {m}   (cfr. [F] Fig. 5-6)', fontsize=10)
    axes[0].legend(fontsize=6.5, ncol=2)

    for T, k in chiavi.items():
        a = np.nan_to_num(ACF_STORE[k][0] - np.nanmean(ACF_STORE[k][0]))
        F = np.abs(np.fft.rfft(a))
        axes[1].plot(np.arange(len(F)), F, lw=1.2, ms=3, markevery=8,
                     label=f'$T$={T:g}', **stile_T(T, TEMPERATURE))
    axes[1].set_xlabel('frequenza'); axes[1].set_ylabel(r'$|\mathrm{FFT}|$')
    axes[1].set_title('Trasformata dell\'ACF: i picchi sono la fase periodica',
                      fontsize=10)
    axes[1].legend(fontsize=6.5, ncol=2)
    plt.tight_layout(); salva(f'12_acf_{m.replace(".", "").replace("-", "_")}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17.5, 4.4))
plot_vs_T(DACF, 'max_fft', r"$\max|\mathrm{FFT}|$ dell'ACF",
          'Transizione periodico → non periodico\n[F] la colloca a $T\\approx0.8$',
          axes[0], UMANO_ACF.get('max_fft'))
axes[0].axhline(CFG.SOGLIA_FFT, color='crimson', ls='--', lw=1.2)
axes[0].text(axes[0].get_xlim()[0], CFG.SOGLIA_FFT * 1.05,
             ' soglia di periodicità', fontsize=7, color='crimson', va='bottom')
axes[0].axvspan(0.7, 1.0, color='orange', alpha=.10)
axes[0].set_yscale('log')

plot_vs_T(DACF, 'gapelmaper', 'GAPELMAPER',
          'Legge di potenza vs esponenziale\n[F] §2.6', axes[1],
          UMANO_ACF.get('gapelmaper'))
axes[1].axhline(1.0, color='k', lw=1.2)
axes[1].text(axes[1].get_xlim()[0], 1.02, ' >1: esponenziale (amorfo)',
             fontsize=7, va='bottom')
axes[1].text(axes[1].get_xlim()[0], 0.98, ' <1: legge di potenza (critico)',
             fontsize=7, va='top')

plot_vs_T(DACF, 'copertura', 'quota di parole con vettore GloVe',
          'Validità della misura\nsotto la soglia l\'ACF è rumore', axes[2],
          UMANO_ACF.get('copertura'))
axes[2].axhline(CFG.COPERTURA_MIN, color='crimson', ls='--', lw=1.4)
axes[2].set_ylim(-.05, 1.05)
plt.tight_layout(); salva('13_transizioni_fase')

# --- diagramma di fase --------------------------------------------------------
ordine = ['solido (periodico)', 'critico (power law)', 'gas (amorfo)',
          'non valutabile', 'n.d.']
colori_fase = {'solido (periodico)': '#4363d8', 'critico (power law)': '#f58231',
               'gas (amorfo)': '#e6194B', 'non valutabile': '#bbbbbb', 'n.d.': '#eeeeee'}
mods = [m for m in MODELLI if m in set(DACF.modello)]
fig, axes = plt.subplots(1, len(mods), figsize=(4.3 * len(mods), 3.8), squeeze=False)
for j, m in enumerate(mods):
    ax = axes[0, j]
    d = DACF[DACF.modello == m]
    ct = (d.groupby(['temperatura', 'fase']).size().unstack(fill_value=0)
          .reindex(columns=[c for c in ordine if c in set(d.fase)], fill_value=0))
    ct = ct.div(ct.sum(1), axis=0)
    bottom = np.zeros(len(ct))
    for c in ct.columns:
        ax.bar(ct.index.astype(str), ct[c].values, bottom=bottom,
               color=colori_fase[c], label=c, width=.82)
        bottom += ct[c].values
    ax.set_title(m, fontsize=9.5); ax.set_xlabel('$T$'); ax.grid(False)
    ax.tick_params(axis='x', labelsize=7)
    if j == 0:
        ax.set_ylabel('quota di documenti')
    if j == len(mods) - 1:
        ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.suptitle('Diagramma di fase per modello — classificazione di [F]', fontsize=11)
plt.tight_layout(); salva('14_diagramma_fase')

print("Composizione delle fasi per modello:")
display(DACF.pivot_table(index='temperatura', columns='fase', values='sample_id',
                         aggfunc='count').fillna(0).astype(int))

### 11.1 Come si presentano i testi nelle tre fasi

[F] apre l'analisi con esempi di testo alle diverse temperature (Fig. 1–4).
Vale la pena guardare i testi che stanno dietro ai numeri.

In [ ]:
def estratto(r, n=380, offset=40000):
    t = r['_testo']
    off = min(offset, max(0, len(t) - n))
    return ' '.join(t[off:off + n].split())

modello_esempio = MODELLI[0]
print(f"Estratti da {modello_esempio}, carattere ~40 000 di ogni documento")
print("=" * 78)
for T in TEMPERATURE:
    c = [r for r in DOCS if r['_modello'] == modello_esempio
         and r['temperature'] == T and r['_valido']]
    if not c:
        continue
    f = DACF[(DACF.modello == modello_esempio) & (DACF.temperatura == T)]['fase']
    moda = f.mode()
    fase = moda.iloc[0] if len(moda) else 'n.d.'
    print(f"\n--- T = {T:g}   [{fase}]")
    print(estratto(c[0]))
print("\n" + "=" * 78)
print(f"\n--- testo umano (War and Peace, continuazione del prompt)")
print(' '.join(UMANO_TXT[40000:40380].split()))

## 12. Sintesi — dove cade la temperatura critica

Ogni paper fornisce almeno un criterio per individuare la temperatura alla quale
il testo generato assomiglia di più a quello umano. Se criteri indipendenti
convergono sullo stesso valore, l'interpretazione in termini di transizione di
fase regge; se divergono, la divergenza stessa è un risultato — [Z] §4.3 la
osserva per i modelli Llama grandi, dove Zipf e Heaps danno temperature diverse.

In [ ]:
def T_estremo(df, col, come):
    """come: 'min', 'max', ('vicino', valore_umano) oppure ('soglia', valore).

    Il modo 'soglia' restituisce la temperatura più bassa alla quale la media
    scende sotto il valore dato. Serve per il parametro d'ordine di [F]: la
    transizione dalla fase periodica è la CADUTA di max|FFT| sotto la soglia,
    non il suo minimo, che cadrebbe sempre all'estremo dello sweep."""
    out = {}
    for m in sorted(set(df.modello)):
        g = df[df.modello == m].groupby('temperatura')[col].mean().dropna()
        if len(g) < 3:
            continue
        if come == 'min':
            out[m] = g.idxmin()
        elif come == 'max':
            out[m] = g.idxmax()
        elif come[0] == 'soglia':
            sotto = g[g < come[1]]
            if len(sotto):
                out[m] = sotto.index.min()
        else:
            rif = come[1]
            if not np.isfinite(rif):
                continue
            out[m] = (g - rif).abs().idxmin()
    return out

CRITERI = [
    ('[Z] MAPE minimo',                 DF,   'MAPE',        'min'),
    ('[Z] $\\alpha$ più vicino a umano', DF,   'alpha',       ('vicino', UMANO['alpha'])),
    ('[Z] $\\beta$ più vicino a umano',  DF,   'beta',        ('vicino', UMANO['beta'])),
    ('[Z] $R$ più vicino a umano',       DF,   'R',           ('vicino', UMANO['R'])),
    ('[L] $D_s$ minima da umano',        DF,   'D_s_umano',   'min'),
    ('[L] $\\alpha_{DFA}$ vicino a umano', DF, 'alpha_DFA',   ('vicino', UMANO['alpha_DFA'])),
    ('[C] $R_{gzip}$ vicino a umano',    DF,   'R_gzip',      ('vicino', UMANO['R_gzip'])),
    ('[F] fine della fase periodica',    DACF, 'max_fft',     ('soglia', CFG.SOGLIA_FFT)),
]

righe = []
for nome, df, col, come in CRITERI:
    if not len(df) or col not in df.columns:
        continue
    for m, T in T_estremo(df, col, come).items():
        righe.append(dict(criterio=nome, modello=m, T_critica=T,
                          params_b=CFG.MODELLI[m]['params_b'],
                          famiglia=CFG.MODELLI[m]['famiglia'],
                          arch=CFG.MODELLI[m]['arch']))
DTC = pd.DataFrame(righe)
DTC.to_csv(CFG.OUT / 'temperature_critiche.csv', index=False)

piv = DTC.pivot_table(index='criterio', columns='modello', values='T_critica')
piv = piv.reindex(columns=[m for m in MODELLI if m in piv.columns])
display(piv.round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
ax = axes[0]
criteri_lista = list(piv.index)
for i, m in enumerate(MODELLI):
    d = DTC[DTC.modello == m]
    if not len(d):
        continue
    y = [criteri_lista.index(c) + (i - len(MODELLI) / 2) * 0.13 for c in d.criterio]
    st = stile_modello(m)
    ax.scatter(d.T_critica, y, s=85, alpha=.85, label=m,
               color=st['color'], marker=st['marker'])
ax.set_yticks(range(len(criteri_lista)))
ax.set_yticklabels(criteri_lista, fontsize=8)
med = DTC.T_critica.median()
ax.axvline(med, color='crimson', ls='--', lw=1.5, label=f'mediana $T^*$ = {med:g}')
ax.axvspan(0.7, 1.0, color='orange', alpha=.10)
ax.set_xlabel('temperatura critica $T^*$')
ax.set_title('Convergenza fra criteri indipendenti', fontsize=10)
ax.legend(fontsize=7, loc='lower right')

ax = axes[1]
for i, nome in enumerate(criteri_lista):
    d = DTC[DTC.criterio == nome]
    ax.plot(d.params_b, d.T_critica, ls='none', ms=9, alpha=.85,
            marker=MARKER_T[i % len(MARKER_T)],
            color=COLORI_T[i % len(COLORI_T)], label=nome)
ax.set_xscale('log'); ax.set_xlabel('parametri [miliardi]')
ax.set_ylabel('temperatura critica $T^*$')
ax.set_title('$T^*$ vs dimensione del modello\n[Z] §4.3: non dovrebbe dipenderne',
             fontsize=10)
ax.legend(fontsize=6.5)
for m in MODELLI:
    ax.annotate(m.replace('Qwen', ''), (CFG.MODELLI[m]['params_b'], ax.get_ylim()[0]),
                fontsize=6.5, rotation=90, va='bottom', ha='center', alpha=.65)
plt.tight_layout(); salva('15_temperatura_critica')

print(f"Mediana di T* su tutti i criteri e modelli: {med:g}")
print(f"Intervallo: [{DTC.T_critica.min():g}, {DTC.T_critica.max():g}]\n")
print("Dispersione per modello:")
display(DTC.groupby('modello')['T_critica'].agg(['count', 'median', 'min', 'max']))
print("\nDispersione per criterio:")
display(DTC.groupby('criterio')['T_critica'].agg(['count', 'median', 'min', 'max']))

if HAS_SCIPY and DTC.modello.nunique() >= 3:
    d = DTC.dropna(subset=['params_b', 'T_critica'])
    rho, pv = _sps.spearmanr(d.params_b, d.T_critica)
    print(f"\nCorrelazione fra dimensione del modello e T*: rho = {rho:+.3f}, p = {pv:.3f}")
    print("  -> " + ("nessuna dipendenza significativa dalla dimensione, "
                     "coerente con [Z] §4.3" if pv > .05 else
                     "dipendenza significativa dalla dimensione, in contrasto con [Z] §4.3"))

print("\nRiferimenti dalla letteratura:")
print("  [Z] Qwen e Granite base, generazione da singolo token: massimo di R e")
print("      minimo del MAPE attorno a t = 1.0-1.2")
print("  [F] transizione da periodico ad amorfo a T ~ 0.8, stato critico fra 0.7 e 1.0")
print("  [L] LSTM su corpus Dickens: ottimo attorno a T = 0.9-1.0 su tutte le metriche")

In [ ]:
# =====================================================================
# 12.1 Pannello complessivo e tabelle
# =====================================================================
METRICHE = [
    ('MAPE',               'MAPE di Zipf',                       DF),
    ('alpha',              r'$\alpha$ (Zipf)',                   DF),
    ('beta',               r'$\beta$ (Heaps)',                   DF),
    ('R',                  '$R$',                                DF),
    ('TTR',                'TTR (token)',                        DF),
    ('alpha_DFA',          r'$\alpha_{\mathrm{DFA}}$',           DF),
    ('R_gzip',             r'$R_{\mathrm{gzip}}$',               DF),
    ('compr_condizionata', 'compressione condizionata',          DF),
    ('H_bit_char',         'entropia [bit/char]',                DF),
    ('D_s_umano',          r'$D_s$ da umano',                    DF),
    ('rip_8gram',          '8-grammi ripetuti',                  DF),
    ('quota_non_ascii',    'quota non-ASCII',                    DF),
    ('max_fft',            r'$\max|\mathrm{FFT}|$',              DACF),
    ('gapelmaper',         'GAPELMAPER',                         DACF),
    ('copertura',          'copertura GloVe',                    DACF),
]
UMANI = {**UMANO, **UMANO_ACF}
attive = [(c, l, d) for c, l, d in METRICHE if len(d) and c in d.columns]
nrow = int(np.ceil(len(attive) / 5))
fig, axes = plt.subplots(nrow, 5, figsize=(23, 3.6 * nrow), squeeze=False)
for i, (col, lab, df) in enumerate(attive):
    ax = axes[i // 5, i % 5]
    plot_vs_T(df, col, lab, lab, ax, UMANI.get(col), legenda=(i == 0))
for j in range(len(attive), nrow * 5):
    axes[j // 5, j % 5].axis('off')
plt.suptitle('Tutte le metriche in funzione della temperatura, con la baseline '
             'umana alla stessa lunghezza', fontsize=13)
plt.tight_layout(); salva('16_pannello_completo')

In [ ]:
# --- tabella riassuntiva -----------------------------------------------------
COL_DF = ['n_token', 'alpha', 'MAPE', 'beta', 'R', 'TTR', 'alpha_DFA',
          'alpha_DFA_mescolato', 'R_gzip', 'compr_condizionata', 'prefix_trend',
          'NCD_shuffle', 'H_bit_char', 'H_parola_norm', 'D_s_umano', 'LCS_umano',
          'rip_8gram', 'dist_rip_media', 'quota_non_ascii']
tab = DF.groupby(['modello', 'temperatura'])[COL_DF].mean().round(4)
if len(DACF):
    tab = tab.join(DACF.groupby(['modello', 'temperatura'])[
        ['max_fft', 'gapelmaper', 'copertura']].mean().round(4))

riga_umana = pd.DataFrame(
    [{c: round(UMANI[c], 4) for c in tab.columns if c in UMANI and
      isinstance(UMANI[c], (int, float))}],
    index=pd.MultiIndex.from_tuples([('War and Peace (umano)', float('nan'))],
                                    names=['modello', 'temperatura']))
TAB = pd.concat([tab, riga_umana])
TAB.to_csv(CFG.OUT / 'tabella_riassuntiva.csv')
display(TAB)

def to_latex(t, caption, label):
    try:
        return t.to_latex(escape=True, na_rep='--', caption=caption, label=label,
                          float_format='%.3f', longtable=True)
    except Exception:
        return t.to_string()

(CFG.OUT / 'tabella_riassuntiva.tex').write_text(
    to_latex(TAB.round(3),
             'Proprietà statistiche dei testi generati in funzione della temperatura, '
             'per modello, con la baseline umana misurata sullo stesso numero di token.',
             'tab:sweep'), encoding='utf-8')

# --- heatmap per modello ------------------------------------------------------
hm = ['MAPE', 'alpha', 'beta', 'R', 'TTR', 'alpha_DFA', 'R_gzip',
      'H_bit_char', 'rip_8gram', 'D_s_umano']
fig, axes = plt.subplots(1, len(MODELLI), figsize=(4.4 * len(MODELLI), 4.2), squeeze=False)
for j, m in enumerate(MODELLI):
    sub = DF[DF.modello == m].groupby('temperatura')[hm].mean()
    z = (sub - sub.mean()) / (sub.std() + 1e-12)
    ax = axes[0, j]
    im = ax.imshow(z.T.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([f'{t:g}' for t in sub.index], fontsize=7, rotation=90)
    ax.set_yticks(range(len(z.columns)))
    ax.set_yticklabels(z.columns if j == 0 else [''] * len(z.columns), fontsize=8)
    ax.set_xlabel('$T$'); ax.set_title(m, fontsize=9.5); ax.grid(False)
    if j == len(MODELLI) - 1:
        plt.colorbar(im, ax=ax, fraction=.046, label='z-score')
plt.suptitle('Metriche standardizzate per modello: la transizione è la banda '
             'dove i colori si ribaltano', fontsize=11)
plt.tight_layout(); salva('17_heatmap_modelli')

In [ ]:
# =====================================================================
# 12.2 Sintesi testuale e inventario degli output
# =====================================================================
linee = []
A = linee.append
A("Sweep di temperatura su testi generati da LLM — sintesi")
A("=" * 70)
A("")
A(f"Modelli analizzati      : {len(MODELLI)}  ({', '.join(MODELLI)})")
A(f"Documenti validi        : {len(DF)} su {len(DOCS)}")
A(f"Temperature             : da {min(TEMPERATURE):g} a {max(TEMPERATURE):g}, "
  f"{len(TEMPERATURE)} valori")
A(f"Tokenizer di analisi    : {CFG.TOKENIZER}, unico per tutti i modelli")
A(f"Troncamento comune      : {CFG.N_TOK:,} token")
A(f"Baseline umana          : War and Peace, continuazione del prompt, "
  f"{UMANO['n_token']:,} token")
A("")
A("Baseline umana alla lunghezza di lavoro")
A("-" * 70)
for c, lab in [('alpha', 'alpha (Zipf)'), ('MAPE', 'MAPE'), ('beta', 'beta (Heaps)'),
               ('R', 'R'), ('alpha_DFA', 'alpha DFA'), ('R_gzip', 'R gzip'),
               ('H_bit_char', 'entropia [bit/char]')]:
    if c in UMANO and np.isfinite(UMANO[c]):
        A(f"  {lab:24s} {UMANO[c]:.4f}")
A("")
A("Temperatura critica per criterio e modello")
A("-" * 70)
A(piv.round(2).to_string())
A("")
A(f"  mediana su tutti i criteri e modelli : T* = {DTC.T_critica.median():g}")
A(f"  intervallo                           : [{DTC.T_critica.min():g}, "
  f"{DTC.T_critica.max():g}]")
A("")
A("Comportamento alle due estremità dello sweep")
A("-" * 70)
T_lo, T_hi = min(TEMPERATURE), max(TEMPERATURE)
for etichetta, T in (('bassa', T_lo), ('alta', T_hi)):
    d = DF[DF.temperatura == T]
    A(f"  Temperatura {etichetta} (T = {T:g}):")
    A(f"    8-grammi ripetuti   {d.rip_8gram.mean():.3f}   (umano {UMANO['rip_8gram']:.3f})")
    A(f"    R_gzip              {d.R_gzip.mean():.3f}   (umano {UMANO['R_gzip']:.3f})")
    A(f"    R                   {d.R.mean():.3f}   (umano {UMANO['R']:.3f})")
    A(f"    alpha DFA           {d.alpha_DFA.mean():.3f}   (umano {UMANO['alpha_DFA']:.3f})")
    A(f"    entropia [bit/char] {d.H_bit_char.mean():.3f}   (umano {UMANO['H_bit_char']:.3f})")
    A(f"    D_s dall'umano      {d.D_s_umano.mean():.3f}")
    if len(DACF):
        da = DACF[DACF.temperatura == T]
        A(f"    max|FFT| dell'ACF   {da.max_fft.mean():.3f}")
        A(f"    copertura GloVe     {da.copertura.mean():.3f}")
A("")
A("File prodotti")
A("-" * 70)
for f in sorted(CFG.OUT.glob('*')):
    A(f"  {f.name}")

testo = "\n".join(linee)
(CFG.OUT / 'sintesi.txt').write_text(testo, encoding='utf-8')
print(testo)

In [ ]:
print("=" * 70)
print("ANALISI COMPLETATA")
print("=" * 70)
print(f"\nTutti i dati e le figure sono in una sola cartella:")
print(f"  {CFG.OUT}")
dati = sorted(CFG.OUT.glob('*.csv')) + sorted(CFG.OUT.glob('*.tex')) + \
       sorted(CFG.OUT.glob('*.txt'))
figure = sorted(CFG.OUT.glob('*.png'))
print(f"\nDati e tabelle ({len(dati)}):")
for f in dati:
    print(f"   {f.name}")
print(f"\nFigure ({len(figure)} in PNG, altrettante in PDF):")
for f in figure:
    print(f"   {f.name}")
print(f"\nNessun file è stato scritto fuori da {CFG.QUI}.")

## 13. Note di metodo

**Unità di analisi.** Il livello primario è il token BPE, come argomentato in
[Z] §3: alle temperature alte le sequenze generate non sono interpretabili come
parole e un'analisi lessicale misurerebbe artefatti di segmentazione. Il
tokenizer è unico per tutti i modelli ed esterno alle famiglie che hanno
generato i testi, replicando la scelta di [Z], che analizza con il tokenizer
Mistral testi prodotti da Qwen, Llama, Granite e Mixtral. Cambiare tokenizer
sposta i valori assoluti degli esponenti ma non la posizione dei loro estremi.

**Baseline umana.** Misurata sulla continuazione dello stesso passaggio usato
come prompt e troncata allo stesso numero di token dei testi generati. Il §4.7
mostra che $R$, $\beta$ e MAPE derivano in modo marcato con la lunghezza: usare
i valori pubblicati in [Z], calcolati su opere intere, come riferimento per
testi da 24 000 token introdurrebbe un errore sistematico. Le bande pubblicate
restano nei grafici come contesto, non come termine di confronto.

**Lunghezza dei testi generati.** Tutti i campioni hanno esattamente 24 000
token di completion, quindi non esiste un confounder di lunghezza fra modelli o
fra temperature. Il troncamento a `CFG.N_TOK` serve solo a rendere identico il
numero di token di *analisi*, che varia leggermente perché il tokenizer di
analisi è diverso da quelli di generazione.

**Testi ricuciti.** La generazione avviene in modalità `continuation`: dopo ogni
EOS si riparte dal testo accumulato. Il §5.2 verifica che il numero di riprese
non introduca discontinuità nelle metriche sensibili all'ordine. Il numero
mediano di riprese per temperatura va comunque riportato: un campione ricucito
da molti pezzi non è identico a uno generato di seguito.

**Limiti della sezione sulle fasi.** L'analisi di [F] presuppone che le parole
abbiano un vettore nello spazio di embedding. Alle temperature alte i modelli
qui analizzati producono sequenze multiscript in cui la copertura di GloVe
crolla sotto la metà e l'ampiezza dell'ACF scende al livello del rumore. In quel
regime il GAPELMAPER è formalmente calcolabile ma privo di contenuto, e i casi
vengono marcati come non valutabili: l'assenza di correlazioni misurabili è
essa stessa la firma della fase amorfa, ma non va confusa con una misura di
decadimento esponenziale. Un embedding multilingue, come quello usato in [F],
alzerebbe la copertura senza cambiare la sostanza, perché a quelle temperature
le sequenze sono casuali anche nelle lingue che coprirebbe.

**Cosa non c'è in questo notebook.** Le analisi di burstiness semantica,
traiettorie PCA e surrogati FGN appartengono a una linea diversa (Altmann et al.
2012 e il lavoro correlato sulla burstiness delle keyword) e sono trattate
altrove. Qui compaiono solo misure definite nei quattro paper di riferimento.